# Causal Decision System — annotated walkthrough

Ask a business question in plain English. **Gemini** turns it into a precise
structured causal query, **Pyro** does all of the arithmetic by Monte Carlo
simulation, and **Gemini** narrates the resulting numbers back.

> **The LLM never does math.** It translates language *in* and narrates results
> *out*. Every number in an answer is produced by the Pyro engine.

## The causal graph

```text
    demand             (C_1, continuous)  ─┐
    market_growth      (C_2, continuous)  ─┤
    competitor_active  (C_3, binary)      ─┼──►  U   utility
    D  capacity policy (decision, 0/1/2)  ─┘
```

Four standing assumptions hold everywhere below:

1. Every cause `C_j` is a **root** variable — nothing causes it.
2. The causes are **mutually independent**.
3. `D` is the decision variable: a small discrete set of actions.
4. `U` is **deterministic** given `D` and the causes. There is no utility noise —
   *all* spread in `U` comes from uncertainty about the causes.

The two quantities the engine computes:

```text
    V(d, x) = E[ u(d, x, C_unobserved) ]      value of action d in context x
    d*(x)   = argmax_d V(d, x)                the optimal policy
```

## Notebook map

| Section | What it defines | Depends on |
| --- | --- | --- |
| §0 Setup | Imports, `.env` secrets, the Gemini client | — |
| §1 Cause models | `CauseDistribution` and its three conjugate subclasses, `Intervention` | §0 |
| §2 Decision engine | `CausalDecisionAPI` — sampling, utility evaluation, policy, do-effects | §1 |
| §3 LLM pipeline | Pydantic query schemas, prompts, the LangGraph state machine | §2 |
| §4 Worked example | The capacity-planning configuration, learning, `ask_causal_system` | §1–§3 |
| §5 Demo queries | Six end-to-end questions exercising each route | §4 |
| Appendix | Superseded parallel implementation — **do not run** | — |

## Running it

Per `CLAUDE.md`, everything runs inside WSL2 with `uv` — never PowerShell, never bare `pip`:

```bash
uv sync --extra notebook
uv run python -m ipykernel install --user --name chanakya   # register the kernel once
uv run jupyter lab
```

The saved kernelspec on this notebook is named `chanakya-causal-decision-system (broken)`;
pick the `chanakya` kernel from the kernel picker before running anything.

**Run order matters.** Cells are strictly sequential — §4 cannot execute until §1–§3
have defined their classes. Two cells are *not* idempotent and are flagged where they
appear: §4.2 (fresh priors) and §4.6 (`learn`, which accumulates). Re-running §4.6
without re-running §4.2 trains on the same ten observations twice.

§5 makes live Gemini calls: two per question (one to interpret, one to explain), so it
needs a working `GOOGLE_API_KEY` and network access. §1–§4 are entirely offline apart
from constructing the client object.


## §0 Setup

### 0.1 Imports

A single preamble covering the whole notebook: `torch`/`pyro` for the math,
`langgraph` for the state machine, `dataclasses`/`typing` for the plumbing.

**This cell is redundant.** Every section below re-imports what it needs — §1 repeats
the `torch`/`pyro`/`typing` imports, §3 repeats the `langgraph` and message imports, §4
imports `os` and `torch`. Nothing here is the sole provider of any name used later, so
the notebook still runs top-to-bottom if this cell is skipped. Its practical value is as
a fast smoke test: if the environment is wrong, this fails in a second rather than
thirty lines into a class definition.

`from __future__ import annotations` defers annotation evaluation. On Python 3.12 the
`X | Y` and `dict[str, Any]` forms used below are native, so it is stylistic rather than
load-bearing — which is why the split cells in §1 and §3 remain valid on their own.


In [26]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable, Literal, TypedDict

import math
import torch
import pyro
import pyro.distributions as dist

from langgraph.graph import StateGraph, START, END


### 0.2 Secrets and the Gemini client

`load_dotenv()` reads a **git-ignored** `.env` from the project root; the guard below it
fails immediately with a clear message rather than letting an authentication error
surface later from inside a LangGraph node.

```text
.env  ->  GOOGLE_API_KEY=your-key-here
```

Per `CLAUDE.md`, keys are never written into the notebook. (`CORE_KNOWLEDGE.md` still
carries a stale warning that this cell hardcodes a key — it no longer does.)

**§4.1 rebuilds `model`, and that is the instance the application actually uses.** What
survives from this cell is the environment check and the loaded variables. Both cells
build the same client: `gemini-3.x` flash uses fixed sampling defaults, so there is no
temperature to pin.


In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage

# Load GOOGLE_API_KEY (and any other secrets) from a git-ignored .env file.
# Create a .env in the project root containing:  GOOGLE_API_KEY=your-key-here
load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError(
        "GOOGLE_API_KEY is not set. Add it to a .env file in the project root."
    )

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)


---

# §1 Layer 1 — Bayesian cause distributions

Each cause owns its own marginal distribution and knows how to learn from data and how
to generate samples. Every model here is **conjugate**, so learning is closed-form and
exact — a few arithmetic updates to the hyperparameters. There is no SVI, no gradient
fitting, no optimiser. Pyro is used purely as a sampler.

| Class | Variable type | Prior → likelihood | Learned parameters | Used in §4? |
| --- | --- | --- | --- | --- |
| `BayesianBernoulliCause` | binary `{0, 1}` | `Beta` → `Bernoulli` | `alpha`, `beta` | yes — `competitor_active` |
| `BayesianCategoricalCause` | `K` classes | `Dirichlet` → `Categorical` | `concentration` (length `K`) | no — available, unused |
| `BayesianNormalCause` | continuous | `Normal-Inverse-Gamma` → `Normal` | `mu`, `kappa`, `alpha`, `beta` | yes — `demand`, `market_growth` |

**Posterior predictive, not posterior.** Each drawn "world" redraws the *parameters*
first and then the *value*. A world is therefore an i.i.d. draw from the posterior
predictive distribution, which carries parameter uncertainty forward into the utility
rather than freezing the parameters at a point estimate. This is what makes the plain
`std / sqrt(n)` standard error in §2 valid.

### 1.1 Imports and type aliases

`Decision = Hashable` — a decision is anything usable as a dict key. §4 uses `0/1/2`,
but strings would work equally well *at this layer* (§3.2 narrows it to `int`).

`InterventionMode` is the whole of the fixed-vs-reoptimised distinction expressed as a
type; §1.6 and §2 give it meaning.


In [28]:
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any, Callable, Hashable, Literal

import math

import pyro
import pyro.distributions as dist
import torch

# ============================================================
# BASIC TYPES
# ============================================================

Tensor = torch.Tensor
Decision = Hashable

InterventionMode = Literal[
    "fixed_policy",
    "reoptimise_policy",
]


### 1.2 `CauseDistribution` — the interface

The contract every cause must satisfy. Four abstract methods, each with a distinct
consumer:

| Method | Must do | Called by |
| --- | --- | --- |
| `update(observations)` | Fold data into the posterior **in place** | `CausalDecisionAPI.learn` (§2) |
| `pyro_sample(num_samples)` | Return a 1-D float tensor of length `num_samples` | `_pyro_world_model` (§2) |
| `posterior_summary()` | Hyperparameters plus interpretable moments | the `posterior_summary` route (§3) |
| `schema()` | Metadata describing the variable to the LLM | `describe_graph()` → the interpreter prompt (§3.5) |

Two invariants that subclasses must respect and the base class cannot enforce:

- **Sample-site names must be globally unique.** Each subclass prefixes its Pyro sites
  with `self.name` (`demand__variance`, `demand__mean`, `demand__value`). Two causes
  sharing a name would collide inside one trace.
- **`update` is cumulative.** It mutates the posterior rather than replacing it, so
  calling it twice with the same data counts that data twice. This is correct Bayesian
  behaviour — and a footgun in a notebook, where re-running a cell is a reflex. See §4.6.


In [29]:
class CauseDistribution(ABC):
    """
    Base class for a learnable marginal distribution associated
    with one independent causal variable C_j.

    Each cause must support:

    1. Updating its posterior from observations.
    2. Posterior-predictive sampling through Pyro.
    
    3. Returning a summary of its learned posterior.
    """

    def __init__(
        self,
        name: str,
        description: str = "",
    ) -> None:
        self.name = name
        self.description = description

    @abstractmethod
    def update(
        self,
        observations: list[Any],
    ) -> None:
        """
        Update the posterior distribution from observations.
        """
        raise NotImplementedError

    @abstractmethod
    def pyro_sample(
        self,
        num_samples: int,
    ) -> Tensor:
        """
        Draw posterior-predictive samples using Pyro.
        """
        raise NotImplementedError

    @abstractmethod
    def posterior_summary(
        self,
    ) -> dict[str, Any]:
        """
        Return posterior hyperparameters and useful moments.
        """
        raise NotImplementedError

    @abstractmethod
    def schema(
        self,
    ) -> dict[str, Any]:
        """
        Return metadata that can be shown to the LLM.
        """
        raise NotImplementedError


### 1.3 `BayesianBernoulliCause` — Beta-Bernoulli

The binary cause. In §4 this is `competitor_active`.

```text
    theta ~ Beta(alpha, beta)
    C | theta ~ Bernoulli(theta)
```

**Update.** After `s` ones and `f` zeros the posterior is `Beta(alpha + s, beta + f)`.
The prior is readable as pseudo-counts: `Beta(2, 3)` says "as if I had already seen 2
active periods and 3 inactive ones" — a weak prior centred on `2/5 = 0.4`. `update`
rejects anything that is not exactly `0` or `1`.

**Reported moments.**

```text
    E[theta]   = alpha / (alpha + beta)
    Var[theta] = alpha * beta / ((alpha + beta)^2 * (alpha + beta + 1))
```

Note that this variance describes uncertainty about the *rate*, not the variance of the
0/1 draws themselves (which is `theta * (1 - theta)`).

**Sampling.** Two stages, both of length `num_samples`: every world draws its own
`theta` and then its own `C`. `.expand([n]).to_event(1)` turns a scalar distribution
into one sample site with event shape `(n,)` — a single named site holding `n`
independent draws, which keeps the Pyro trace flat. The returned tensor is `float`
(`0.0`/`1.0`), which is what the utility function in §4.4 multiplies against.


In [30]:
class BayesianBernoulliCause(CauseDistribution):
    """
    Binary causal variable.

        theta ~ Beta(alpha, beta)
        C | theta ~ Bernoulli(theta)

    After observing s ones and f zeros:

        theta | data ~ Beta(alpha + s, beta + f)
    """

    def __init__(
        self,
        name: str,
        alpha: float = 1.0,
        beta: float = 1.0,
        description: str = "",
    ) -> None:
        super().__init__(
            name=name,
            description=description,
        )

        if alpha <= 0 or beta <= 0:
            raise ValueError(
                "Beta prior parameters must be positive."
            )

        self.alpha = float(alpha)
        self.beta = float(beta)

    def update(
        self,
        observations: list[int | float],
    ) -> None:
        if not observations:
            return

        values = torch.as_tensor(
            observations,
            dtype=torch.float32,
        )

        valid = torch.logical_or(
            values == 0,
            values == 1,
        )

        if not bool(valid.all()):
            raise ValueError(
                f"Cause '{self.name}' only accepts 0/1 observations."
            )

        successes = float(values.sum())
        failures = float(values.numel()) - successes

        self.alpha += successes
        self.beta += failures

    def pyro_sample(
        self,
        num_samples: int,
    ) -> Tensor:
        """
        For every Monte Carlo world m:

            theta^(m) ~ posterior Beta
            C^(m) ~ Bernoulli(theta^(m))
        """

        alpha = torch.tensor(
            self.alpha,
            dtype=torch.float32,
        )

        beta = torch.tensor(
            self.beta,
            dtype=torch.float32,
        )

        probabilities = pyro.sample(
            f"{self.name}__probability",
            dist.Beta(alpha, beta)
            .expand([num_samples])
            .to_event(1),
        )

        values = pyro.sample(
            f"{self.name}__value",
            dist.Bernoulli(probabilities).to_event(1),
        )

        return values

    def posterior_summary(
        self,
    ) -> dict[str, Any]:
        total = self.alpha + self.beta

        posterior_mean = self.alpha / total

        posterior_variance = (
            self.alpha
            * self.beta
            / (
                total**2
                * (total + 1.0)
            )
        )

        return {
            "distribution": "Beta-Bernoulli",
            "alpha": self.alpha,
            "beta": self.beta,
            "posterior_probability_mean": posterior_mean,
            "posterior_probability_variance":
                posterior_variance,
        }

    def schema(
        self,
    ) -> dict[str, Any]:
        return {
            "name": self.name,
            "type": "binary",
            "allowed_values": [0, 1],
            "description": self.description,
        }


### 1.4 `BayesianCategoricalCause` — Dirichlet-Categorical

**Defined but unused.** No cause in the §4 configuration is categorical; this class is
here so a multi-class variable (region, tier, season) can be dropped in without touching
Layer 2.

```text
    theta ~ Dirichlet(concentration)      # length K
    C | theta ~ Categorical(theta)
```

**Update.** Count the observations per category and add the counts to `concentration` —
the same pseudo-count logic as Beta-Bernoulli, generalised to `K` classes. Observations
may be given either as labels (`"north"`) or as integer indexes (`0`); `_convert_observation`
maps and range-checks both, so a typo in a label raises rather than silently miscounting.

Three things to know before using it:

- **Categories are integers on the wire.** `pyro_sample` returns the category *index* as
  a float, so a utility function has to compare `region == 2.0`, not `region == "north"`.
  `schema()` ships the index→label mapping to the LLM so the model can do that
  translation when it fills in `context`.
- `ParsedCausalQuery.context` in §3.3 is typed `dict[str, float]`, so an LLM-supplied
  context value must already be the index.
- Unlike the other two classes the value site is not wrapped in `.to_event(1)` — it does
  not need to be, since `Categorical` over a `(n, K)` probability tensor already yields a
  batch of `n` draws. The returned shape is `(n,)`, matching the others.


In [31]:
class BayesianCategoricalCause(CauseDistribution):
    """
    Categorical causal variable.

        theta ~ Dirichlet(alpha)
        C | theta ~ Categorical(theta)

    Categories are represented internally by integer indexes:

        0, 1, ..., K - 1

    Human-readable category labels can also be supplied.
    """

    def __init__(
        self,
        name: str,
        categories: list[str],
        concentration: list[float] | None = None,
        description: str = "",
    ) -> None:
        super().__init__(
            name=name,
            description=description,
        )

        if len(categories) < 2:
            raise ValueError(
                "A categorical cause requires at least two categories."
            )

        if len(set(categories)) != len(categories):
            raise ValueError(
                "Category labels must be unique."
            )

        self.categories = list(categories)

        if concentration is None:
            concentration = [1.0] * len(categories)

        if len(concentration) != len(categories):
            raise ValueError(
                "There must be one concentration parameter "
                "for each category."
            )

        if any(value <= 0 for value in concentration):
            raise ValueError(
                "Dirichlet concentration parameters must be positive."
            )

        self.concentration = torch.tensor(
            concentration,
            dtype=torch.float32,
        )

        self.category_to_index = {
            category: index
            for index, category in enumerate(categories)
        }

    def _convert_observation(
        self,
        value: int | str,
    ) -> int:
        if isinstance(value, str):
            if value not in self.category_to_index:
                raise ValueError(
                    f"Unknown category '{value}' for cause "
                    f"'{self.name}'."
                )

            return self.category_to_index[value]

        index = int(value)

        if index < 0 or index >= len(self.categories):
            raise ValueError(
                f"Category index {index} is invalid for "
                f"cause '{self.name}'."
            )

        return index

    def update(
        self,
        observations: list[int | str],
    ) -> None:
        if not observations:
            return

        indexes = torch.tensor(
            [
                self._convert_observation(value)
                for value in observations
            ],
            dtype=torch.long,
        )

        counts = torch.bincount(
            indexes,
            minlength=len(self.categories),
        ).float()

        self.concentration = (
            self.concentration + counts
        )

    def pyro_sample(
        self,
        num_samples: int,
    ) -> Tensor:
        probabilities = pyro.sample(
            f"{self.name}__probabilities",
            dist.Dirichlet(self.concentration)
            .expand([num_samples])
            .to_event(1),
        )

        values = pyro.sample(
            f"{self.name}__value",
            dist.Categorical(probabilities),
        )

        return values.float()

    def posterior_summary(
        self,
    ) -> dict[str, Any]:
        probabilities = (
            self.concentration
            / self.concentration.sum()
        )

        return {
            "distribution": "Dirichlet-Categorical",
            "categories": self.categories,
            "concentration":
                self.concentration.tolist(),
            "posterior_probabilities": {
                category: float(probabilities[index])
                for index, category
                in enumerate(self.categories)
            },
        }

    def schema(
        self,
    ) -> dict[str, Any]:
        return {
            "name": self.name,
            "type": "categorical",
            "allowed_values": list(
                range(len(self.categories))
            ),
            "category_labels": {
                index: category
                for index, category
                in enumerate(self.categories)
            },
            "description": self.description,
        }


### 1.5 `BayesianNormalCause` — Normal-Inverse-Gamma

The continuous cause, with **both mean and variance unknown**. In §4 this is `demand`
and `market_growth`.

```text
    sigma^2      ~ InverseGamma(alpha, beta)
    mu | sigma^2 ~ Normal(mu_0, sigma^2 / kappa)
    C | mu, sigma^2 ~ Normal(mu, sigma^2)
```

**The four hyperparameters, and how to read them:**

| Parameter | Meaning | Prior strength interpretation |
| --- | --- | --- |
| `mu` | posterior mean of the mean | the centre |
| `kappa` | confidence in `mu` | ≈ number of pseudo-observations behind the mean |
| `alpha` | shape of the variance posterior | ≈ half the pseudo-observations behind the variance |
| `beta` | scale of the variance posterior | pseudo sum-of-squares |

**The exact conjugate update** for `n` observations with sample mean `x̄`:

```text
    kappa_n = kappa + n
    mu_n    = (kappa * mu + n * x̄) / kappa_n
    alpha_n = alpha + n / 2
    beta_n  = beta + 0.5 * SUM (x_i - x̄)^2  +  kappa * n * (x̄ - mu)^2 / (2 * kappa_n)
```

That last term is the one worth understanding: the variance grows not only from the
spread *within* the data but also from the gap between the data and what the prior
expected. Data that is tight but far from `mu` still increases `beta`.

**Sampling — three stages, in this order:** variance → mean → value. Drawing the mean
*conditional on the drawn variance* is what makes this a true posterior predictive; its
analytic marginal is a Student-t, which is why extreme worlds appear slightly more often
than a Normal at the posterior mean would produce.

**`expected_sampling_variance` is `E[sigma^2] = beta / (alpha - 1)`** (infinite for
`alpha <= 1`, hence the guard). That is the variance of a *single observation given the
parameters* — the spread of the predictive draws is wider, by a factor of
`(1 + 1/kappa)`, because the mean is uncertain too.

**Support is unbounded.** Nothing constrains a draw to be positive, so a
`demand` world can come out negative. §4.4 discusses what that does to the utility.


In [32]:
class BayesianNormalCause(CauseDistribution):
    """
    Continuous causal variable with unknown mean and variance.

        C_i | mu, sigma^2 ~ Normal(mu, sigma^2)

        sigma^2 ~ InverseGamma(alpha, beta)

        mu | sigma^2
            ~ Normal(mu_0, sigma^2 / kappa)

    The class stores the current posterior hyperparameters:

        mu
        kappa
        alpha
        beta

    Updates are exact conjugate Bayesian updates.
    Posterior-predictive samples are drawn through Pyro.
    """

    def __init__(
        self,
        name: str,
        mu: float = 0.0,
        kappa: float = 1.0,
        alpha: float = 2.0,
        beta: float = 2.0,
        description: str = "",
    ) -> None:
        super().__init__(
            name=name,
            description=description,
        )

        if kappa <= 0:
            raise ValueError(
                "kappa must be positive."
            )

        if alpha <= 0 or beta <= 0:
            raise ValueError(
                "Inverse-Gamma parameters must be positive."
            )

        self.mu = float(mu)
        self.kappa = float(kappa)
        self.alpha = float(alpha)
        self.beta = float(beta)

    def update(
        self,
        observations: list[int | float],
    ) -> None:
        if not observations:
            return

        values = torch.as_tensor(
            observations,
            dtype=torch.float32,
        )

        n = int(values.numel())
        sample_mean = values.mean()

        centred_sum_squares = (
            (values - sample_mean) ** 2
        ).sum()

        old_mu = torch.tensor(
            self.mu,
            dtype=torch.float32,
        )

        old_kappa = torch.tensor(
            self.kappa,
            dtype=torch.float32,
        )

        old_alpha = torch.tensor(
            self.alpha,
            dtype=torch.float32,
        )

        old_beta = torch.tensor(
            self.beta,
            dtype=torch.float32,
        )

        new_kappa = old_kappa + n

        new_mu = (
            old_kappa * old_mu
            + n * sample_mean
        ) / new_kappa

        new_alpha = old_alpha + n / 2.0

        mean_difference_adjustment = (
            old_kappa
            * n
            * (sample_mean - old_mu) ** 2
            / (2.0 * new_kappa)
        )

        new_beta = (
            old_beta
            + 0.5 * centred_sum_squares
            + mean_difference_adjustment
        )

        self.mu = float(new_mu)
        self.kappa = float(new_kappa)
        self.alpha = float(new_alpha)
        self.beta = float(new_beta)

    def pyro_sample(
        self,
        num_samples: int,
    ) -> Tensor:
        """
        For each simulated world:

            sigma^2 ~ InverseGamma(alpha, beta)

            mu ~ Normal(
                posterior_mu,
                sqrt(sigma^2 / kappa)
            )

            C ~ Normal(mu, sqrt(sigma^2))
        """

        alpha = torch.tensor(
            self.alpha,
            dtype=torch.float32,
        )

        beta = torch.tensor(
            self.beta,
            dtype=torch.float32,
        )

        posterior_mu = torch.tensor(
            self.mu,
            dtype=torch.float32,
        )

        kappa = torch.tensor(
            self.kappa,
            dtype=torch.float32,
        )

        variance = pyro.sample(
            f"{self.name}__variance",
            dist.InverseGamma(alpha, beta)
            .expand([num_samples])
            .to_event(1),
        )

        sampled_mean = pyro.sample(
            f"{self.name}__mean",
            dist.Normal(
                posterior_mu.expand(num_samples),
                torch.sqrt(variance / kappa),
            ).to_event(1),
        )

        values = pyro.sample(
            f"{self.name}__value",
            dist.Normal(
                sampled_mean,
                torch.sqrt(variance),
            ).to_event(1),
        )

        return values

    def posterior_summary(
        self,
    ) -> dict[str, Any]:
        if self.alpha > 1:
            expected_variance = (
                self.beta
                / (self.alpha - 1.0)
            )
        else:
            expected_variance = math.inf

        return {
            "distribution":
                "Normal-Inverse-Gamma",
            "posterior_mean": self.mu,
            "kappa": self.kappa,
            "alpha": self.alpha,
            "beta": self.beta,
            "expected_sampling_variance":
                expected_variance,
        }

    def schema(
        self,
    ) -> dict[str, Any]:
        return {
            "name": self.name,
            "type": "continuous",
            "description": self.description,
        }


### 1.6 `Intervention` — the `do(...)` request object

A frozen dataclass describing one causal contrast: `do(variable = value)`, optionally
against `do(variable = baseline_value)`.

| Field | Meaning |
| --- | --- |
| `variable` | which cause to clamp — must be a cause, never `D` |
| `value` | the value forced under the intervention |
| `baseline_value` | the comparison arm; `None` has a special meaning (below) |
| `mode` | `fixed_policy` or `reoptimise_policy` |
| `fixed_decision` | required when `mode == "fixed_policy"`, ignored otherwise |

**The two modes answer genuinely different questions:**

```text
fixed_policy       E[U | do(C=c1), D=d]  -  E[U | do(C=c0), D=d]
                   "Holding my current plan, what does this change cost me?"

reoptimise_policy  max_d E[U | do(C=c1), D=d]  -  max_d E[U | do(C=c0), D=d]
                   "If I adapt my plan optimally, what does this change cost me?"
```

The reoptimised effect is never worse than the fixed one, because re-optimising can
only help. The gap between them is the value of being able to adapt.

**`baseline_value=None` changes the estimand.** The baseline arm then runs with *no*
intervention at all, so the comparison is `do(C=c)` against `C` left stochastic at its
learned posterior — an intervention-versus-status-quo contrast, not a contrast of two
interventions. That is often what a user means ("what if we forced it to 1?"), but it is
a different quantity from `do(C=1)` vs `do(C=0)`, and the §3.5 prompt is careful to only
fill `baseline_value` when the user actually names a comparison value.

Validation of `fixed_decision` lives in §2 and §3.2, not here — the dataclass is frozen
but not self-validating.


In [33]:
@dataclass(frozen=True)
class Intervention:
    """
    Represents a query involving:

        do(variable = value)

    baseline_value:
        If supplied, compare do(variable=value) against
        do(variable=baseline_value).

        If absent, compare the intervention against the ordinary
        observational model where that cause remains stochastic.

    mode:
        fixed_policy:
            Hold D fixed.

        reoptimise_policy:
            Recalculate the optimal action under each condition.
    """

    variable: str
    value: float

    baseline_value: float | None = None

    mode: InterventionMode = "reoptimise_policy"

    fixed_decision: Decision | None = None


---

# §2 Layer 2 — `CausalDecisionAPI`, the math engine

One class holding everything Layer 3 is allowed to call. It is constructed from the
causes (§1), a list of decisions, and a deterministic `utility_function(decision, causes)
-> Tensor`.

### Method map

| Method | Answers | Returns |
| --- | --- | --- |
| `describe_graph()` | "what does this model contain?" | metadata dict — also the payload injected into the LLM prompt |
| `learn(observations)` | "fold this data in" | posterior summaries after updating |
| `posterior_summaries()` | "what has been learned?" | per-cause summary dicts |
| `sample_causal_worlds(...)` | — (internal) | `{cause_name: tensor(num_samples)}` |
| `evaluate_decision(d, worlds)` | "what is `U` under `d` in each world?" | one utility per world |
| `evaluate_all_decisions(...)` | "how do all actions compare?" | per-action statistics + P(best) |
| `optimal_policy(...)` | "what should I do?" | the argmax plus the full comparison |
| `intervention_effect(...)` | "what does forcing `C` to `c` do?" | a causal contrast in one of two modes |

### How `do(...)` is implemented

`_pyro_world_model` walks the causes and picks one of three cases per cause:

```text
    name in interventions  ->  clamp to a constant   do(C = c)
    name in context        ->  clamp to a constant   observed X = x
    neither                ->  cause.pyro_sample()   stays stochastic
```

**Notice that the first two branches are identical code.** That is not an oversight —
it is a property of this graph. Every cause is a root with no parents, so there is
nothing to cut: `P(U | do(C=c), D=d)` and `P(U | C=c, D=d)` are the same distribution
here. The intervention/observation distinction is kept in the *interface* because it is
the honest description of what the user asked, and because the moment any cause gains a
parent (or a confounder) the two branches must diverge. Where the distinction *does*
already bite numerically is the fixed-vs-reoptimised policy axis of `intervention_effect`.

A contradiction — the same variable observed as one value and intervened to another —
raises rather than silently preferring one. Consistent duplicates are allowed.

### Common random numbers

`evaluate_all_decisions` samples the worlds **once** and evaluates **every** decision
against **the same** worlds. Comparisons between decisions then differ only by the
decision, not by sampling noise, which sharply reduces the variance of the *difference*.
`intervention_effect` extends the same trick across arms by passing the identical `seed`
to both, making the contrast paired.

The practical consequence: the reported `monte_carlo_standard_error` is correct for each
decision's expected utility on its own, but **conservative for comparisons** — the true
standard error of a difference between two decisions is smaller than the two individual
errors suggest, because their errors are positively correlated by construction.

### Reported statistics

| Field | Meaning |
| --- | --- |
| `expected_utility` | mean of `U` over worlds — the decision-theoretic quantity |
| `utility_standard_deviation` | spread of `U` across worlds — the *risk*, not the estimation error |
| `monte_carlo_standard_error` | `std / sqrt(n)` — precision of the mean estimate; shrink it by raising `num_samples` |
| `mean_lower_95` / `mean_upper_95` | 95% interval **on the mean**, not on `U` itself |
| `probability_each_action_is_best` | fraction of worlds in which that action wins the argmax |

`probability_each_action_is_best` is a different question from `expected_utility`: an
action can have the highest average payoff while winning in a minority of worlds. Ties
in `argmax` go to the lowest index, which is immaterial for continuous utilities.

`seed` is passed to `pyro.set_rng_seed` on every call, so results are reproducible and
two calls with the same seed and sample count are directly comparable.

**The utility function's shape contract is enforced.** It must return exactly one value
per world; anything else raises with a message naming both sizes. This is the check that
catches a utility function which accidentally reduces over the batch.

The class is long by design — it is a single coherent object and cannot be split across
cells without resorting to monkey-patching. The `# -----` comment rules inside it mark
the same sections listed in the table above.


In [34]:
class CausalDecisionAPI:
    """
    Causal structure:

        C_1  ──┐
        C_2  ──┤
        ...    ├──> U
        C_p  ──┤
        D    ──┘

    Assumptions:

    1. Every C_j is a root cause.
    2. The causes are mutually independent.
    3. D is the decision variable.
    4. U is deterministic given D and C.
    5. Each cause has its own learnable marginal distribution.

    The expected value of decision d given observed context x is:

        V(d, x)
        =
        E[
            u(d, x, C_unobserved)
        ]

    The optimal policy at context x is:

        d*(x) = argmax_d V(d, x)
    """

    def __init__(
        self,
        causes: dict[str, CauseDistribution],
        decisions: list[Decision],
        utility_function: Callable[
            [Decision, dict[str, Tensor]],
            Tensor,
        ],
        decision_descriptions:
            dict[Decision, str] | None = None,
        utility_description: str = "",
    ) -> None:
        if not causes:
            raise ValueError(
                "At least one cause is required."
            )

        if not decisions:
            raise ValueError(
                "At least one decision is required."
            )

        duplicate_names = (
            len(causes) != len(set(causes))
        )

        if duplicate_names:
            raise ValueError(
                "Cause names must be unique."
            )

        for name, cause in causes.items():
            if name != cause.name:
                raise ValueError(
                    f"Dictionary key '{name}' does not match "
                    f"cause name '{cause.name}'."
                )

        self.causes = causes
        self.decisions = list(decisions)
        self.utility_function = utility_function

        self.decision_descriptions = (
            decision_descriptions
            or {
                decision: str(decision)
                for decision in decisions
            }
        )

        self.utility_description = utility_description

    # --------------------------------------------------------
    # MODEL DESCRIPTION
    # --------------------------------------------------------

    def describe_graph(
        self,
    ) -> dict[str, Any]:
        """
        Metadata used both by developers and by the LLM query
        interpreter.
        """

        return {
            "causes": {
                name: cause.schema()
                for name, cause
                in self.causes.items()
            },
            "decision_variable": "D",
            "decisions": {
                str(decision):
                    self.decision_descriptions.get(
                        decision,
                        str(decision),
                    )
                for decision in self.decisions
            },
            "utility_variable": "U",
            "utility_description":
                self.utility_description,
            "assumptions": [
                "All causes are mutually independent.",
                "All causes are parents of utility.",
                "The decision is a parent of utility.",
                "Utility is deterministic given the decision and causes.",
            ],
        }

    # --------------------------------------------------------
    # LEARNING
    # --------------------------------------------------------

    def learn(
        self,
        observations:
            dict[str, list[Any]],
    ) -> dict[str, Any]:
        """
        Update every supplied marginal distribution.

        Example:

            api.learn(
                {
                    "demand": [10, 11, 12],
                    "competitor_active": [0, 1, 0],
                }
            )
        """

        unknown = (
            set(observations)
            - set(self.causes)
        )

        if unknown:
            raise ValueError(
                f"Unknown causes in training data: "
                f"{sorted(unknown)}"
            )

        for name, values in observations.items():
            self.causes[name].update(values)

        return self.posterior_summaries()

    def posterior_summaries(
        self,
    ) -> dict[str, Any]:
        return {
            name: cause.posterior_summary()
            for name, cause
            in self.causes.items()
        }

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    def _validate_context(
        self,
        context: dict[str, float] | None,
    ) -> None:
        context = context or {}

        unknown = (
            set(context)
            - set(self.causes)
        )

        if unknown:
            raise ValueError(
                f"Unknown context variables: "
                f"{sorted(unknown)}"
            )

    def _validate_interventions(
        self,
        interventions:
            dict[str, float] | None,
    ) -> None:
        interventions = interventions or {}

        unknown = (
            set(interventions)
            - set(self.causes)
        )

        if unknown:
            raise ValueError(
                f"Unknown intervention variables: "
                f"{sorted(unknown)}"
            )

    # --------------------------------------------------------
    # PYRO CAUSAL MODEL
    # --------------------------------------------------------

    def _pyro_world_model(
        self,
        num_samples: int,
        context:
            dict[str, float] | None = None,
        interventions:
            dict[str, float] | None = None,
    ) -> dict[str, Tensor]:
        """
        Pyro model generating Monte Carlo causal worlds.

        Context variables are observed before D is selected.

        Interventions replace the structural mechanism for that
        variable with a constant:

            do(C_k = c)
        """

        context = context or {}
        interventions = interventions or {}

        sampled_causes: dict[str, Tensor] = {}

        for name, cause in self.causes.items():

            if name in interventions:
                sampled_causes[name] = torch.full(
                    (num_samples,),
                    float(interventions[name]),
                    dtype=torch.float32,
                )

            elif name in context:
                sampled_causes[name] = torch.full(
                    (num_samples,),
                    float(context[name]),
                    dtype=torch.float32,
                )

            else:
                sampled_causes[name] = (
                    cause.pyro_sample(
                        num_samples=num_samples,
                    )
                )

        return sampled_causes

    def sample_causal_worlds(
        self,
        num_samples: int = 10_000,
        context:
            dict[str, float] | None = None,
        interventions:
            dict[str, float] | None = None,
        seed: int = 123,
    ) -> dict[str, Tensor]:
        """
        Execute the Pyro model and return sampled cause values.
        """

        if num_samples < 2:
            raise ValueError(
                "num_samples must be at least 2."
            )

        self._validate_context(context)
        self._validate_interventions(interventions)

        context = context or {}
        interventions = interventions or {}

        contradictory_variables = (
            set(context)
            & set(interventions)
        )

        for variable in contradictory_variables:
            observed_value = float(context[variable])
            intervention_value = float(
                interventions[variable]
            )

            if observed_value != intervention_value:
                raise ValueError(
                    f"'{variable}' is observed as "
                    f"{observed_value} but intervened on as "
                    f"{intervention_value}."
                )

        pyro.set_rng_seed(seed)

        traced_model = pyro.poutine.trace(
            self._pyro_world_model
        )

        trace = traced_model.get_trace(
            num_samples=num_samples,
            context=context,
            interventions=interventions,
        )

        return trace.nodes["_RETURN"]["value"]

    # --------------------------------------------------------
    # UTILITY EVALUATION
    # --------------------------------------------------------

    def evaluate_decision(
        self,
        decision: Decision,
        sampled_causes: dict[str, Tensor],
    ) -> Tensor:
        if decision not in self.decisions:
            raise ValueError(
                f"Unknown decision '{decision}'."
            )

        utility = self.utility_function(
            decision,
            sampled_causes,
        )

        if not isinstance(utility, torch.Tensor):
            utility = torch.as_tensor(
                utility,
                dtype=torch.float32,
            )

        utility = utility.reshape(-1)

        expected_size = next(
            iter(sampled_causes.values())
        ).numel()

        if utility.numel() != expected_size:
            raise ValueError(
                "The utility function must return exactly one "
                "utility value for each simulated causal world. "
                f"Expected {expected_size}, received "
                f"{utility.numel()}."
            )

        return utility.float()

    def evaluate_all_decisions(
        self,
        context:
            dict[str, float] | None = None,
        interventions:
            dict[str, float] | None = None,
        num_samples: int = 10_000,
        seed: int = 123,
    ) -> dict[str, Any]:
        """
        Uses common random numbers:

        The same sampled causal worlds are evaluated under every
        possible decision. This usually reduces Monte Carlo noise
        in comparisons between decisions.
        """

        sampled_causes = self.sample_causal_worlds(
            num_samples=num_samples,
            context=context,
            interventions=interventions,
            seed=seed,
        )

        utility_samples: dict[Decision, Tensor] = {}
        evaluations: list[dict[str, Any]] = []

        for decision in self.decisions:
            utilities = self.evaluate_decision(
                decision=decision,
                sampled_causes=sampled_causes,
            )

            utility_samples[decision] = utilities

            mean = utilities.mean()
            standard_deviation = utilities.std(
                unbiased=True
            )

            standard_error = (
                standard_deviation
                / math.sqrt(num_samples)
            )

            evaluations.append(
                {
                    "decision": decision,
                    "decision_description":
                        self.decision_descriptions.get(
                            decision,
                            str(decision),
                        ),
                    "expected_utility": float(mean),
                    "utility_standard_deviation":
                        float(standard_deviation),
                    "monte_carlo_standard_error":
                        float(standard_error),
                    "mean_lower_95": float(
                        mean
                        - 1.96 * standard_error
                    ),
                    "mean_upper_95": float(
                        mean
                        + 1.96 * standard_error
                    ),
                }
            )

        utility_matrix = torch.stack(
            [
                utility_samples[decision]
                for decision in self.decisions
            ],
            dim=1,
        )

        winning_decision_indexes = (
            utility_matrix.argmax(dim=1)
        )

        probability_best = {
            str(decision): float(
                (
                    winning_decision_indexes
                    == index
                )
                .float()
                .mean()
            )
            for index, decision
            in enumerate(self.decisions)
        }

        return {
            "context": context or {},
            "interventions":
                interventions or {},
            "num_samples": num_samples,
            "action_evaluations": evaluations,
            "probability_each_action_is_best":
                probability_best,
        }

    # --------------------------------------------------------
    # OPTIMAL POLICY
    # --------------------------------------------------------

    def optimal_policy(
        self,
        context:
            dict[str, float] | None = None,
        interventions:
            dict[str, float] | None = None,
        num_samples: int = 10_000,
        seed: int = 123,
    ) -> dict[str, Any]:
        """
        Calculate:

            d*(x)
            =
            argmax_d E[U | D=d, X=x]

        optionally under an intervention.
        """

        evaluation = self.evaluate_all_decisions(
            context=context,
            interventions=interventions,
            num_samples=num_samples,
            seed=seed,
        )

        optimal_evaluation = max(
            evaluation["action_evaluations"],
            key=lambda row: row[
                "expected_utility"
            ],
        )

        return {
            "query_type": "optimal_policy",
            "optimal_decision":
                optimal_evaluation["decision"],
            "optimal_decision_description":
                optimal_evaluation[
                    "decision_description"
                ],
            "optimal_expected_utility":
                optimal_evaluation[
                    "expected_utility"
                ],
            **evaluation,
        }

    # --------------------------------------------------------
    # INTERVENTION EFFECT
    # --------------------------------------------------------

    def intervention_effect(
        self,
        intervention: Intervention,
        context:
            dict[str, float] | None = None,
        num_samples: int = 10_000,
        seed: int = 123,
    ) -> dict[str, Any]:
        """
        Supports two causal estimands.

        Fixed-policy effect:

            E[U | do(C_k=c_1), D=d]
            -
            E[U | do(C_k=c_0), D=d]

        Reoptimised-policy effect:

            max_d E[U | do(C_k=c_1), D=d]
            -
            max_d E[U | do(C_k=c_0), D=d]
        """

        if intervention.variable not in self.causes:
            raise ValueError(
                f"Unknown intervention variable "
                f"'{intervention.variable}'."
            )

        if intervention.mode == "fixed_policy":
            if intervention.fixed_decision is None:
                raise ValueError(
                    "fixed_policy mode requires "
                    "fixed_decision."
                )

            if (
                intervention.fixed_decision
                not in self.decisions
            ):
                raise ValueError(
                    f"Unknown fixed decision "
                    f"'{intervention.fixed_decision}'."
                )

        target_interventions = {
            intervention.variable:
                intervention.value
        }

        baseline_interventions = (
            {
                intervention.variable:
                    intervention.baseline_value
            }
            if intervention.baseline_value
            is not None
            else {}
        )

        if (
            intervention.mode
            == "reoptimise_policy"
        ):
            target = self.optimal_policy(
                context=context,
                interventions=target_interventions,
                num_samples=num_samples,
                seed=seed,
            )

            baseline = self.optimal_policy(
                context=context,
                interventions=baseline_interventions,
                num_samples=num_samples,
                seed=seed,
            )

            effect = (
                target["optimal_expected_utility"]
                - baseline[
                    "optimal_expected_utility"
                ]
            )

            return {
                "query_type":
                    "intervention_effect",
                "estimand":
                    "reoptimised_policy_effect",
                "intervention_variable":
                    intervention.variable,
                "intervention_value":
                    intervention.value,
                "baseline_value":
                    intervention.baseline_value,
                "target_optimal_decision":
                    target["optimal_decision"],
                "target_optimal_decision_description":
                    target[
                        "optimal_decision_description"
                    ],
                "baseline_optimal_decision":
                    baseline["optimal_decision"],
                "baseline_optimal_decision_description":
                    baseline[
                        "optimal_decision_description"
                    ],
                "target_expected_utility":
                    target[
                        "optimal_expected_utility"
                    ],
                "baseline_expected_utility":
                    baseline[
                        "optimal_expected_utility"
                    ],
                "causal_effect": effect,
                "target_result": target,
                "baseline_result": baseline,
            }

        fixed_decision = (
            intervention.fixed_decision
        )

        target_evaluation = (
            self.evaluate_all_decisions(
                context=context,
                interventions=target_interventions,
                num_samples=num_samples,
                seed=seed,
            )
        )

        baseline_evaluation = (
            self.evaluate_all_decisions(
                context=context,
                interventions=baseline_interventions,
                num_samples=num_samples,
                seed=seed,
            )
        )

        target_row = next(
            row
            for row in target_evaluation[
                "action_evaluations"
            ]
            if row["decision"] == fixed_decision
        )

        baseline_row = next(
            row
            for row in baseline_evaluation[
                "action_evaluations"
            ]
            if row["decision"] == fixed_decision
        )

        effect = (
            target_row["expected_utility"]
            - baseline_row["expected_utility"]
        )

        return {
            "query_type":
                "intervention_effect",
            "estimand":
                "fixed_policy_effect",
            "fixed_decision":
                fixed_decision,
            "fixed_decision_description":
                self.decision_descriptions.get(
                    fixed_decision,
                    str(fixed_decision),
                ),
            "intervention_variable":
                intervention.variable,
            "intervention_value":
                intervention.value,
            "baseline_value":
                intervention.baseline_value,
            "target_expected_utility":
                target_row["expected_utility"],
            "baseline_expected_utility":
                baseline_row[
                    "expected_utility"
                ],
            "causal_effect": effect,
            "target_evaluation": target_row,
            "baseline_evaluation": baseline_row,
        }


---

# §3 Layer 3 — the LangGraph + Gemini pipeline

The natural-language front end. The design rule holds throughout: **the LLM translates,
the engine computes.**

```mermaid
graph TD
    START --> interpret_query
    interpret_query --> validate_query
    validate_query -->|optimal_policy| optimal_policy
    validate_query -->|intervention_effect| intervention_effect
    validate_query -->|posterior_summary| posterior_summary
    validate_query -->|clarification| clarification
    validate_query -->|error| error
    optimal_policy --> explain_result
    intervention_effect --> explain_result
    posterior_summary --> explain_result
    clarification --> END
    explain_result --> END
    error --> END
```

Two LLM calls per question, at the two ends, with deterministic Python in between:

```text
   English  ─►  interpret  ─►  validate  ─►  Pyro math  ─►  explain  ─►  English
              (Gemini)      (pure Python)   (authoritative)  (Gemini)
```

`validate_query` is the safety net. Structured output constrains the LLM's *shape*, but
nothing stops it inventing a plausible-sounding variable name — so every name it produces
is re-checked against the real graph before a single sample is drawn.

### 3.1 Imports

Self-contained, so this section runs whether or not §0.1 did. `pydantic` supplies the
structured-output schemas; `json` serialises the graph into the prompt and the results
back out to the explainer.


In [35]:
from __future__ import annotations

from typing import Any, Literal, TypedDict

import json

from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
)
from langgraph.graph import (
    END,
    START,
    StateGraph,
)
from pydantic import (
    BaseModel,
    Field,
    model_validator,
)


### 3.2 `ParsedIntervention` — the intervention schema

What the LLM is allowed to say about an intervention. Every `description=` string is
prompt surface: it is sent to Gemini as part of the tool schema, so these strings are
instructions, not comments.

| Field | Required | Notes |
| --- | --- | --- |
| `variable` | yes | a cause name; validated against the real graph in §3.6 |
| `value` | yes | the value to force |
| `baseline_value` | no | `None` ⇒ compare against the stochastic model (§1.6) |
| `mode` | yes | `fixed_policy` or `reoptimise_policy` |
| `fixed_decision` | conditionally | required when `mode == "fixed_policy"` |

The `model_validator` enforces the one cross-field rule — `fixed_policy` without a
`fixed_decision` is incoherent. A Pydantic failure here is caught in §3.6's
`interpret_query` and turned into an error answer rather than an exception.

**`fixed_decision` is typed `int | None`**, which is narrower than Layer 1's
`Decision = Hashable`. It matches the `0/1/2` decisions of §4, but a configuration with
string-labelled actions would need this widened.


In [36]:
class ParsedIntervention(BaseModel):
    """
    Intervention extracted from natural-language user input.
    """

    variable: str = Field(
        description=(
            "Name of the non-decision causal variable "
            "being intervened upon."
        )
    )

    value: float = Field(
        description=(
            "Value imposed by do(variable=value)."
        )
    )

    baseline_value: float | None = Field(
        default=None,
        description=(
            "Optional comparison intervention value. "
            "For example, compare do(C=1) with do(C=0). "
            "Leave null when comparing against the ordinary "
            "stochastic model."
        ),
    )

    mode: Literal[
        "fixed_policy",
        "reoptimise_policy",
    ] = Field(
        description=(
            "Use fixed_policy when the user wants to hold "
            "one decision fixed. Use reoptimise_policy when "
            "the optimal decision should be recalculated."
        )
    )

    fixed_decision: int | None = Field(
        default=None,
        description=(
            "Decision to hold fixed. Required only when "
            "mode is fixed_policy."
        ),
    )

    @model_validator(mode="after")
    def validate_intervention(
        self,
    ) -> "ParsedIntervention":
        if (
            self.mode == "fixed_policy"
            and self.fixed_decision is None
        ):
            raise ValueError(
                "fixed_policy requires fixed_decision."
            )

        return self


### 3.3 `ParsedCausalQuery` — the query schema

The single object the interpreter must produce. `query_type` selects the route:

| `query_type` | Meaning | Runs |
| --- | --- | --- |
| `optimal_policy` | "what should I do given what I know?" | `optimal_policy` (§2) |
| `intervention_effect` | "what would forcing `C` to `c` do?" | `intervention_effect` (§2) |
| `posterior_summary` | "what has the system learned?" | `posterior_summaries()` — no sampling |
| `clarification_required` | essential information is missing | asks a follow-up, no math |

`context` holds only values the user **actually stated**. Omission is meaningful:
anything absent stays stochastic and is integrated over. An LLM that helpfully fills in a
"typical" demand would silently convert an honest expectation over uncertainty into a
confident point answer — which is why the §3.5 prompt says *never invent values* four
separate times.

`context: dict[str, float]` means every context value must be numeric, so a categorical
cause (§1.4) has to arrive as its index.

The validators reject two incoherent combinations: `intervention_effect` without an
intervention, and `clarification_required` without a question.


In [37]:
class ParsedCausalQuery(BaseModel):
    """
    Structured interpretation of a natural-language query.
    """

    query_type: Literal[
        "optimal_policy",
        "intervention_effect",
        "posterior_summary",
        "clarification_required",
    ] = Field(
        description=(
            "The operation requested by the user."
        )
    )

    context: dict[str, float] = Field(
        default_factory=dict,
        description=(
            "Cause values observed before choosing D. "
            "Only include values explicitly stated or clearly "
            "given by the user. Unknown variables must not be "
            "included."
        ),
    )

    intervention: ParsedIntervention | None = Field(
        default=None,
        description=(
            "Intervention specification for intervention queries."
        ),
    )

    clarification_question: str | None = Field(
        default=None,
        description=(
            "Question to ask when essential information is missing."
        ),
    )

    @model_validator(mode="after")
    def validate_query(
        self,
    ) -> "ParsedCausalQuery":
        if (
            self.query_type
            == "intervention_effect"
            and self.intervention is None
        ):
            raise ValueError(
                "intervention_effect requires an intervention."
            )

        if (
            self.query_type
            == "clarification_required"
            and not self.clarification_question
        ):
            raise ValueError(
                "clarification_required needs a question."
            )

        return self


### 3.4 `CausalApplicationState` — the graph's state

The channel dictionary passed between nodes. `total=False` means every key is optional,
so a node returns only the keys it wants to change and LangGraph merges that partial
update into the running state.

| Key | Written by | Read by |
| --- | --- | --- |
| `user_input` | the caller | `interpret_query`, `explain_result` |
| `parsed_query` | `interpret_query` | `validate_query`, all compute nodes |
| `route` | `validate_query` (or an erroring node) | the conditional edge |
| `numerical_result` | the compute nodes | `explain_result` |
| `final_answer` | `explain_result`, `clarification`, `error` | the caller |
| `error` | any node that fails | the routers, `error` |

No reducers are configured, so a later write to a key replaces an earlier one
(last-writer-wins). There is also **no `messages` channel and no checkpointer** — the
graph is stateless between invocations, which is why a `clarification_required` answer
currently dead-ends rather than continuing a conversation. That is exactly what ticket
CHK-3 in `TICKETS.md` sets out to fix.


In [38]:
class CausalApplicationState(
    TypedDict,
    total=False,
):
    user_input: str

    parsed_query: ParsedCausalQuery

    route: str

    numerical_result: dict[str, Any]

    final_answer: str

    error: str


### 3.5 The two prompts

**`build_query_interpreter_prompt(causal_api)`** — the translator's brief. It embeds
`describe_graph()` as JSON, so the live cause names, types, allowed values, decisions
and assumptions are always in sync with the actual configuration; nothing about the
domain is hardcoded in the prompt text. The rules it teaches:

- *Observation vs intervention.* "X is currently 4" is context; "force X to 4" is
  `do(X = 4)`. The prompt lists the verbs that signal an intervention — set, force,
  intervene, make, impose.
- *Never invent* context values, intervention values, or a fixed decision.
- *Use only names present in the supplied graph* — belt to §3.6's braces.
- *Unknown variables stay stochastic* — silence is a valid, meaningful input.
- *Ask for clarification* only when something essential is genuinely missing, with three
  worked examples of what qualifies.

The prompt is rebuilt on every call, so a graph changed at runtime is picked up
immediately — at the cost of re-serialising the graph JSON per question.

**`build_result_explanation_prompt()`** — the narrator's brief, and mostly a list of
prohibitions: do not recalculate, do not change any number, do not invent intervals or
observations, do not call an observational comparison an intervention, do not imply the
LLM computed the policy. Then eight things the explanation must cover, including which
causes stayed stochastic and whether the policy was fixed or reoptimised — the two
details a plausible-sounding but wrong summary would gloss over.


In [39]:
def build_query_interpreter_prompt(
    causal_api: CausalDecisionAPI,
) -> str:
    graph_description = (
        causal_api.describe_graph()
    )

    return f"""
You are the query interpreter for a causal decision system.

Your task is to translate the user's natural-language request into
one structured query. You do not perform numerical calculations.

CAUSAL GRAPH
============

{json.dumps(graph_description, indent=2)}

The graph contains:

    independent causes C_1, ..., C_p
                  |
                  v
    decision D -> utility U

Utility is deterministic given D and all causes.

SUPPORTED OPERATIONS
====================

1. optimal_policy

Use this when the user wants the decision D that maximises expected
utility given the information currently observed.

Observed cause values belong in `context`.

Examples:

- "Demand is 12. Which policy is best?"
- "The competitor is active and growth is 0.04. What should I do?"

Variables that are not observed must be omitted from `context`.
They remain stochastic in the Pyro model.

2. intervention_effect

Use this when the user asks about externally setting or forcing a
cause to a value:

    do(C_k = c)

Phrases such as:

- set
- force
- intervene
- make
- change externally
- impose

normally indicate an intervention rather than an observation.

The intervention variable must be a known cause. It cannot be the
decision variable D.

There are two modes:

fixed_policy:
    Hold one specified decision fixed while comparing expected
    utility under the intervention.

reoptimise_policy:
    Recalculate the optimal decision under the intervention and
    under the baseline condition.

3. posterior_summary

Use this when the user asks what the system has learned about the
cause distributions.

4. clarification_required

Use this only when essential information is missing.

Examples of genuinely missing information:

- The user asks for a fixed-policy intervention but does not say
  which decision to fix.
- The user asks to set a variable but gives no intervention value.
- The mentioned variable could refer to more than one known cause.

IMPORTANT RULES
===============

- Never invent context values.
- Never invent intervention values.
- Never invent a fixed decision.
- Use only cause names present in the supplied graph.
- Use only decision values present in the supplied graph.
- Unknown variables remain stochastic.
- Observation and intervention are conceptually different.
- "X is currently 4" is normally an observation.
- "Force X to 4" is normally an intervention.
- If a baseline value is explicitly supplied, store it.
- If no baseline value is supplied, leave baseline_value null.
"""


def build_result_explanation_prompt() -> str:
    return """
You explain results from a causal decision and Pyro inference engine.

The numerical result supplied to you is authoritative.

Do not:

- recalculate the result;
- change any numerical values;
- invent uncertainty intervals;
- invent observations;
- claim that an observational comparison is an intervention;
- claim that the language model calculated the policy.

Explain clearly:

1. What the user asked.
2. Which causes were observed.
3. Which causes remained stochastic.
4. Whether this was policy optimisation or intervention analysis.
5. The recommended decision or estimated causal effect.
6. The important expected-utility values.
7. Any Monte Carlo uncertainty included in the result.
8. Whether the policy was fixed or reoptimised.

Keep the explanation readable but mathematically accurate.
"""


### 3.6 `build_causal_langgraph_app` — assembling the state machine

A closure over `causal_api`, `model`, `default_num_samples` and `default_seed`, returning
a compiled graph. Every node is defined inside it, so the sample count and seed are baked
into the app at build time rather than passed per query.

| Node | Does | On failure |
| --- | --- | --- |
| `interpret_query` | Gemini + `with_structured_output(ParsedCausalQuery)` | sets `error`, routes to `error` |
| `validate_query` | pure-Python re-check of every LLM-supplied name; sets `route` | sets `error` |
| `optimal_policy` | `causal_api.optimal_policy(context=...)` | sets `error` |
| `intervention_effect` | rebuilds a real `Intervention` (§1.6) and runs it | sets `error` |
| `posterior_summary` | reads `posterior_summaries()` — no sampling, no seed | — |
| `clarification` | returns the LLM's follow-up question verbatim | falls back to a generic prompt |
| `explain_result` | Gemini narrates the authoritative numbers | see below |
| `error` | surfaces `error` as the final answer | — |

**What `validate_query` actually re-checks**, in order: empty/errored input → `error`;
`clarification_required` → short-circuit; unknown context variables → `error`; for
intervention queries, a missing intervention, an unknown intervention variable, and a
`fixed_decision` that is not in `causal_api.decisions`. Only after all of those does it
set `route` to the query type.

Details worth knowing:

- **`explain_result` degrades gracefully.** If the narration call fails, the math has
  already succeeded, so it returns the raw result JSON plus the error rather than losing
  the answer. This is the right failure mode for a system whose numbers are the product.
- **`clarification` and `error` terminate without an LLM call** — they go straight to
  `END`, so a rejected query costs one Gemini call, not two.
- **`run_optimal_policy` passes only `context`, never `interventions`.** An
  `optimal_policy` query carrying an intervention would ignore it silently; routing means
  that combination should not arise, but the guarantee comes from the router, not here.
- **Context and intervention on the same variable conflict.** If a user both observes and
  forces the same variable at different values, §2 raises and the node converts it into
  an error answer.
- **`compile()` takes no checkpointer**, hence the statelessness noted in §3.4.


In [40]:
def build_causal_langgraph_app(
    causal_api: CausalDecisionAPI,
    model: Any,
    default_num_samples: int = 10_000,
    default_seed: int = 123,
):
    """
    Construct and compile the LangGraph application.

    Parameters
    ----------
    causal_api:
        The CausalDecisionAPI built in Step 1.

    model:
        Your ChatGoogleGenerativeAI instance.

    default_num_samples:
        Number of Pyro posterior-predictive worlds used for each
        numerical query.

    default_seed:
        Random seed used for reproducibility.
    """

    structured_query_model = (
        model.with_structured_output(
            ParsedCausalQuery
        )
    )

    # --------------------------------------------------------
    # NODE 1: LLM INTERPRETS NATURAL LANGUAGE
    # --------------------------------------------------------

    def interpret_query(
        state: CausalApplicationState,
    ) -> CausalApplicationState:
        user_input = state.get(
            "user_input",
            "",
        ).strip()

        if not user_input:
            return {
                "error":
                    "The user input is empty.",
                "route": "error",
            }

        try:
            parsed_query = (
                structured_query_model.invoke(
                    [
                        SystemMessage(
                            content=(
                                build_query_interpreter_prompt(
                                    causal_api
                                )
                            )
                        ),
                        HumanMessage(
                            content=user_input
                        ),
                    ]
                )
            )

            return {
                "parsed_query": parsed_query,
            }

        except Exception as exc:
            return {
                "error": (
                    "The LLM could not convert the request "
                    f"into a structured causal query: {exc}"
                ),
                "route": "error",
            }

    # --------------------------------------------------------
    # NODE 2: DETERMINISTIC VALIDATION
    # --------------------------------------------------------

    def validate_query(
        state: CausalApplicationState,
    ) -> CausalApplicationState:
        if state.get("error"):
            return {
                "route": "error",
            }

        parsed = state["parsed_query"]

        if (
            parsed.query_type
            == "clarification_required"
        ):
            return {
                "route": "clarification",
            }

        known_causes = set(
            causal_api.causes
        )

        unknown_context = (
            set(parsed.context)
            - known_causes
        )

        if unknown_context:
            return {
                "error": (
                    "The interpreted query contains unknown "
                    f"context variables: {sorted(unknown_context)}"
                ),
                "route": "error",
            }

        if (
            parsed.query_type
            == "intervention_effect"
        ):
            if parsed.intervention is None:
                return {
                    "error":
                        "The intervention is missing.",
                    "route": "error",
                }

            if (
                parsed.intervention.variable
                not in known_causes
            ):
                return {
                    "error": (
                        "The query attempts to intervene on "
                        f"unknown variable "
                        f"'{parsed.intervention.variable}'."
                    ),
                    "route": "error",
                }

            if (
                parsed.intervention.mode
                == "fixed_policy"
                and parsed.intervention.fixed_decision
                not in causal_api.decisions
            ):
                return {
                    "error": (
                        "The fixed decision is not one of the "
                        f"allowed decisions: "
                        f"{causal_api.decisions}."
                    ),
                    "route": "error",
                }

        return {
            "route": parsed.query_type,
        }

    # --------------------------------------------------------
    # NODE 3A: OPTIMAL POLICY
    # --------------------------------------------------------

    def run_optimal_policy(
        state: CausalApplicationState,
    ) -> CausalApplicationState:
        parsed = state["parsed_query"]

        try:
            result = causal_api.optimal_policy(
                context=parsed.context,
                num_samples=default_num_samples,
                seed=default_seed,
            )

            return {
                "numerical_result": result,
            }

        except Exception as exc:
            return {
                "error": (
                    "Optimal-policy calculation failed: "
                    f"{exc}"
                )
            }

    # --------------------------------------------------------
    # NODE 3B: INTERVENTION EFFECT
    # --------------------------------------------------------

    def run_intervention_effect(
        state: CausalApplicationState,
    ) -> CausalApplicationState:
        parsed = state["parsed_query"]

        if parsed.intervention is None:
            return {
                "error":
                    "No intervention was supplied."
            }

        intervention = Intervention(
            variable=(
                parsed.intervention.variable
            ),
            value=(
                parsed.intervention.value
            ),
            baseline_value=(
                parsed.intervention.baseline_value
            ),
            mode=(
                parsed.intervention.mode
            ),
            fixed_decision=(
                parsed.intervention.fixed_decision
            ),
        )

        try:
            result = (
                causal_api.intervention_effect(
                    intervention=intervention,
                    context=parsed.context,
                    num_samples=
                        default_num_samples,
                    seed=default_seed,
                )
            )

            return {
                "numerical_result": result,
            }

        except Exception as exc:
            return {
                "error": (
                    "Intervention calculation failed: "
                    f"{exc}"
                )
            }

    # --------------------------------------------------------
    # NODE 3C: POSTERIOR SUMMARY
    # --------------------------------------------------------

    def run_posterior_summary(
        state: CausalApplicationState,
    ) -> CausalApplicationState:
        return {
            "numerical_result": {
                "query_type":
                    "posterior_summary",
                "posterior_distributions":
                    causal_api.posterior_summaries(),
            }
        }

    # --------------------------------------------------------
    # NODE 3D: CLARIFICATION
    # --------------------------------------------------------

    def return_clarification(
        state: CausalApplicationState,
    ) -> CausalApplicationState:
        parsed = state["parsed_query"]

        question = (
            parsed.clarification_question
            or (
                "Please provide the missing information "
                "needed to define the causal query."
            )
        )

        return {
            "final_answer": question,
        }

    # --------------------------------------------------------
    # NODE 4: LLM EXPLAINS PYRO RESULT
    # --------------------------------------------------------

    def explain_result(
        state: CausalApplicationState,
    ) -> CausalApplicationState:
        if state.get("error"):
            return {
                "final_answer": state["error"]
            }

        parsed = state["parsed_query"]
        numerical_result = (
            state["numerical_result"]
        )

        payload = {
            "original_user_input":
                state["user_input"],
            "structured_query":
                parsed.model_dump(),
            "authoritative_numerical_result":
                numerical_result,
        }

        try:
            response = model.invoke(
                [
                    SystemMessage(
                        content=(
                            build_result_explanation_prompt()
                        )
                    ),
                    HumanMessage(
                        content=json.dumps(
                            payload,
                            indent=2,
                            default=str,
                        )
                    ),
                ]
            )

            # Gemini 3 returns a list of content blocks (text plus a
            # thought signature), so `.content` is no longer a bare string.
            # `.text` concatenates the text blocks and passes a plain string
            # straight through.
            answer = response.text

            return {
                "final_answer": answer,
            }

        except Exception as exc:
            # The numerical calculation has already succeeded,
            # so return it even if the explanation LLM fails.
            return {
                "final_answer": (
                    "The numerical calculation succeeded, "
                    "but the LLM explanation failed.\n\n"
                    + json.dumps(
                        numerical_result,
                        indent=2,
                        default=str,
                    )
                    + f"\n\nExplanation error: {exc}"
                )
            }

    # --------------------------------------------------------
    # ERROR NODE
    # --------------------------------------------------------

    def return_error(
        state: CausalApplicationState,
    ) -> CausalApplicationState:
        return {
            "final_answer": state.get(
                "error",
                "An unknown application error occurred.",
            )
        }

    # --------------------------------------------------------
    # ROUTING FUNCTIONS
    # --------------------------------------------------------

    def route_after_validation(
        state: CausalApplicationState,
    ) -> str:
        return state.get(
            "route",
            "error",
        )

    def route_after_calculation(
        state: CausalApplicationState,
    ) -> str:
        if state.get("error"):
            return "error"

        return "explain"

    # --------------------------------------------------------
    # CONSTRUCT THE GRAPH
    # --------------------------------------------------------

    graph_builder = StateGraph(
        CausalApplicationState
    )

    graph_builder.add_node(
        "interpret_query",
        interpret_query,
    )

    graph_builder.add_node(
        "validate_query",
        validate_query,
    )

    graph_builder.add_node(
        "optimal_policy",
        run_optimal_policy,
    )

    graph_builder.add_node(
        "intervention_effect",
        run_intervention_effect,
    )

    graph_builder.add_node(
        "posterior_summary",
        run_posterior_summary,
    )

    graph_builder.add_node(
        "clarification",
        return_clarification,
    )

    graph_builder.add_node(
        "explain_result",
        explain_result,
    )

    graph_builder.add_node(
        "error",
        return_error,
    )

    graph_builder.add_edge(
        START,
        "interpret_query",
    )

    graph_builder.add_edge(
        "interpret_query",
        "validate_query",
    )

    graph_builder.add_conditional_edges(
        "validate_query",
        route_after_validation,
        {
            "optimal_policy":
                "optimal_policy",
            "intervention_effect":
                "intervention_effect",
            "posterior_summary":
                "posterior_summary",
            "clarification":
                "clarification",
            "error":
                "error",
        },
    )

    graph_builder.add_conditional_edges(
        "optimal_policy",
        route_after_calculation,
        {
            "explain":
                "explain_result",
            "error":
                "error",
        },
    )

    graph_builder.add_conditional_edges(
        "intervention_effect",
        route_after_calculation,
        {
            "explain":
                "explain_result",
            "error":
                "error",
        },
    )

    graph_builder.add_conditional_edges(
        "posterior_summary",
        route_after_calculation,
        {
            "explain":
                "explain_result",
            "error":
                "error",
        },
    )

    graph_builder.add_edge(
        "explain_result",
        END,
    )

    graph_builder.add_edge(
        "clarification",
        END,
    )

    graph_builder.add_edge(
        "error",
        END,
    )

    return graph_builder.compile()


---

# §4 Worked example — capacity planning

A firm chooses how much capacity to build. Too little and it leaves demand unserved; too
much and it pays for idle capacity. Three uncertain quantities drive the outcome.

### Data dictionary

| Variable | Model | Type / unit | Prior | Prior mean | Role in the utility |
| --- | --- | --- | --- | --- | --- |
| `demand` | `BayesianNormalCause` | continuous, demand units | `mu=10, kappa=1, alpha=3, beta=8` | 10.0 | scales revenue via `effective_demand` |
| `market_growth` | `BayesianNormalCause` | continuous, decimal rate (`0.05` = 5%) | `mu=0.02, kappa=2, alpha=4, beta=0.02` | 0.02 | multiplies demand |
| `competitor_active` | `BayesianBernoulliCause` | binary `{0, 1}` | `Beta(2, 3)` | 0.4 | subtracts a competitive loss |
| `D` | decision | `{0, 1, 2}` | — | — | fixes capacity, margin, fixed cost, penalty |
| `U` | deterministic | profit-like reward | — | — | the objective |

The priors are deliberately weak: `demand`'s `kappa=1` is worth a single pseudo-observation,
so the ten data points in §4.6 dominate it immediately.

### 4.1 The Gemini client

This rebinds the `model` name from §0.2; from here on, `model` means this one. There is
no `temperature=0` here: the `gemini-3.x` flash models ship fixed sampling defaults and
the API ignores a supplied temperature (LangChain warns when you pass one). Determinism
for the schema-extraction call comes from `with_structured_output()` constraining the
response, not from a sampling knob.

`os.getenv("GOOGLE_API_KEY")` depends on §0.2 having run `load_dotenv()`. If it returns
`None` the client is constructed anyway and fails later at call time, which is precisely
why §0.2's explicit guard exists.


In [41]:
import os

import torch

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
)

# ============================================================
# 1. YOUR GEMINI MODEL
# ============================================================

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.getenv(
        "GOOGLE_API_KEY"
    ),
)


### 4.2 The causes

Three fresh cause objects with their priors. **This cell is the reset point**: running it
discards whatever was learned and restores the priors in the table above. Anything below
that mutates state (§4.6) should be re-run from here, not on its own.

Reading the priors:

- **`demand`** — `mu=10` with `kappa=1` (one pseudo-observation, so barely anchored) and
  `alpha=3, beta=8` giving `E[sigma^2] = 8/2 = 4`, i.e. a prior spread of about ±2 units.
- **`market_growth`** — `mu=0.02` with `kappa=2`, and `alpha=4, beta=0.02` giving
  `E[sigma^2] ≈ 0.0067`, about ±8 percentage points. Wide enough that negative growth is
  entirely ordinary.
- **`competitor_active`** — `Beta(2, 3)`, mean `0.4`, worth five pseudo-observations.

The `description=` strings are not decoration: `schema()` forwards them into the
interpreter prompt, and they are how Gemini learns that `market_growth` is a decimal
rather than a percentage. A user saying "growth is 5%" gets `0.05` because that mapping is
spelled out here.


In [42]:
causes = {
    "demand": BayesianNormalCause(
        name="demand",
        mu=10.0,
        kappa=1.0,
        alpha=3.0,
        beta=8.0,
        description=(
            "Current customer demand measured in demand units."
        ),
    ),

    "market_growth": BayesianNormalCause(
        name="market_growth",
        mu=0.02,
        kappa=2.0,
        alpha=4.0,
        beta=0.02,
        description=(
            "Market growth rate represented as a decimal. "
            "For example, 0.05 means five percent growth."
        ),
    ),

    "competitor_active": BayesianBernoulliCause(
        name="competitor_active",
        alpha=2.0,
        beta=3.0,
        description=(
            "Whether a competing company is actively running "
            "a strong campaign. Zero means inactive and one "
            "means active."
        ),
    ),
}


### 4.3 The decisions

Three capacity policies, indexed `0`, `1`, `2`. `decision_descriptions` carries the
human-readable label into `describe_graph()` (so the LLM can map "the aggressive option"
onto `2`) and back out into every result dict (so the explanation can name the action
rather than print an integer).


In [43]:
decisions = [
    0,
    1,
    2,
]

decision_descriptions = {
    0: "Conservative capacity policy",
    1: "Balanced capacity policy",
    2: "Aggressive capacity policy",
}


### 4.4 The utility function

The structural equation `U = u(D, demand, market_growth, competitor_active)`. It is
deterministic and fully vectorised — it receives tensors of length `num_samples` and must
return one utility per world (§2 enforces the shape).

**Per-decision economics:**

| `D` | Capacity | Margin/unit | Fixed cost | Idle-capacity penalty |
| --- | --- | --- | --- | --- |
| 0 — conservative | 8 | 6 | 8 | 0.5 |
| 1 — balanced | 14 | 7 | 18 | 1.2 |
| 2 — aggressive | 22 | 8 | 35 | 2.5 |

Bigger capacity buys a better margin (scale economics) but costs more up front and is
punished harder when it goes unused.

**The four terms:**

```text
    effective_demand = demand * (1 + market_growth)
    units_sold       = min(effective_demand, capacity)          # capacity is a hard ceiling

    revenue          = margin_per_unit * units_sold
    competitive_loss = competitor_active * 2 * max(effective_demand - 5, 0)
    capacity_penalty = unused_capacity_penalty * max(capacity - effective_demand, 0)

    U = revenue - fixed_cost - competitive_loss - capacity_penalty
```

**`competitive_loss` does not depend on `D`.** That has a consequence worth stating
plainly, because it shapes what §5's demo queries can possibly show: since the competitor
term is identical across all three actions *within the same world*, it shifts every
action's utility by the same amount and **can never change which action is best**. So in
this configuration the reoptimised-policy effect of `competitor_active` equals its
fixed-policy effect, and the optimal decision is invariant to it. That is a property of
this utility function, not of the engine — making the competitor term interact with
capacity (say, scaling the loss by unused capacity) would break the symmetry and make the
two modes diverge. A good first modification if you want to see the modes come apart.

**A worked world.** With `demand = 13` and `market_growth = 0.02`, `effective_demand = 13.26`:

| `D` | `units_sold` | revenue | − fixed | − idle penalty | `U` (no competitor) |
| --- | --- | --- | --- | --- | --- |
| 0 | 8.00 | 48.00 | 8 | 0.00 | **40.00** |
| 1 | 13.26 | 92.82 | 18 | 0.89 | **73.93** |
| 2 | 13.26 | 106.08 | 35 | 21.85 | **49.23** |

Decision 1 wins: 0 is capacity-starved and 2 is paying for 8.7 idle units. An active
competitor subtracts `2 * (13.26 - 5) = 16.52` from all three, leaving the ranking intact.
These are single-world arithmetic, *not* the Monte Carlo answer — the engine averages over
the full posterior for `market_growth` and `competitor_active`, which shifts the numbers.

**Caveat: the Normal causes are unbounded.** A sufficiently negative draw makes
`effective_demand` negative, and nothing clamps it — `units_sold` goes negative, revenue
goes negative, and the idle penalty is charged on more than the full capacity. With the
learned posteriors this is far into the tail, but a heavier-tailed or lower-mean
configuration would need `torch.clamp(effective_demand, min=0.0)`.

Note the parameter named `causes` shadows the module-level `causes` dict inside this
function's body — deliberate and harmless, since it only ever wants the passed-in worlds.


In [44]:
def utility_function(
    decision: int,
    causes: dict[str, torch.Tensor],
) -> torch.Tensor:
    """
    Deterministic structural equation:

        U = u(D, demand, market_growth, competitor_active)

    There is no additional random utility noise.

    Any uncertainty in U comes from uncertainty in the causes.
    """

    demand = causes["demand"]

    market_growth = causes[
        "market_growth"
    ]

    competitor_active = causes[
        "competitor_active"
    ]

    if decision == 0:
        capacity = 8.0
        margin_per_unit = 6.0
        fixed_cost = 8.0
        unused_capacity_penalty = 0.5

    elif decision == 1:
        capacity = 14.0
        margin_per_unit = 7.0
        fixed_cost = 18.0
        unused_capacity_penalty = 1.2

    elif decision == 2:
        capacity = 22.0
        margin_per_unit = 8.0
        fixed_cost = 35.0
        unused_capacity_penalty = 2.5

    else:
        raise ValueError(
            f"Unknown decision: {decision}"
        )

    effective_demand = (
        demand
        * (1.0 + market_growth)
    )

    capacity_tensor = torch.full_like(
        effective_demand,
        capacity,
    )

    units_sold = torch.minimum(
        effective_demand,
        capacity_tensor,
    )

    revenue = (
        margin_per_unit
        * units_sold
    )

    competitive_loss = (
        competitor_active
        * 2.0
        * torch.clamp(
            effective_demand - 5.0,
            min=0.0,
        )
    )

    unused_capacity = torch.clamp(
        capacity_tensor
        - effective_demand,
        min=0.0,
    )

    capacity_penalty = (
        unused_capacity_penalty
        * unused_capacity
    )

    utility = (
        revenue
        - fixed_cost
        - competitive_loss
        - capacity_penalty
    )

    return utility


### 4.5 Constructing the engine

Wires §4.2–§4.4 into a `CausalDecisionAPI`. The constructor validates that the dict keys
match each cause's own `name` — the check that catches a copy-paste error where a cause is
filed under the wrong key.

`utility_description` is free text for the LLM only; it appears in `describe_graph()` and
helps the interpreter judge which action a vaguely worded question is about.


In [45]:
causal_api = CausalDecisionAPI(
    causes=causes,
    decisions=decisions,
    utility_function=utility_function,
    decision_descriptions=
        decision_descriptions,
    utility_description=(
        "Profit-like reward equal to revenue minus fixed cost, "
        "competitive loss and unused-capacity penalty."
    ),
)


### 4.6 Learning from data

Ten historical observations per variable, folded into the priors by exact conjugate
update. Read the arrays as three columns of one table:

| # | `demand` | `market_growth` | `competitor_active` |
| --- | --- | --- | --- |
| 1 | 9.5 | 0.01 | 0 |
| 2 | 11.0 | 0.03 | 1 |
| 3 | 12.5 | −0.01 | 0 |
| 4 | 10.8 | 0.04 | 0 |
| 5 | 13.2 | 0.02 | 1 |
| 6 | 8.9 | 0.01 | 0 |
| 7 | 11.7 | 0.05 | 1 |
| 8 | 12.1 | 0.00 | 0 |
| 9 | 10.2 | 0.02 | 0 |
| 10 | 14.0 | 0.03 | 1 |
| | mean **11.39** | mean **0.02** | **4** of 10 active |

The rows are not paired observations of one period — each cause is independent, so `learn`
treats the three columns as three separate one-dimensional samples.

**Resulting posteriors** (exact, before float32 rounding):

| Cause | Prior | Posterior | Reading |
| --- | --- | --- | --- |
| `demand` | `mu=10, kappa=1, alpha=3, beta=8` | `mu≈11.264, kappa=11, alpha=8, beta≈20.683` | mean pulled from 10 → 11.26; `E[sigma^2]≈2.95`, predictive sd ≈ **1.80** |
| `market_growth` | `mu=0.02, kappa=2, alpha=4, beta=0.02` | `mu=0.02, kappa=12, alpha=9, beta=0.0215` | sample mean equalled the prior mean, so only confidence moved; predictive sd ≈ **0.054** |
| `competitor_active` | `Beta(2, 3)` | `Beta(6, 9)` | `P(active)` 0.4 → **0.4**, now on 15 pseudo-observations instead of 5 |

Two coincidences make this data unusually clean: `market_growth`'s sample mean is exactly
the prior mean (so the `(x̄ - mu)^2` term vanishes and `mu` does not move), and the
competitor rate `4/10` lands on the prior's `0.4`. Only the *confidence* changed for those
two. `demand` is the one that genuinely moved.

Note that `market_growth`'s predictive sd of ~0.054 is more than twice its mean — negative
growth worlds are common, which is exactly the uncertainty the decision is being made
under.

> **⚠ Not idempotent.** `learn` *accumulates*. Running this cell twice applies the same
> ten observations twice — `demand`'s `kappa` would go to 21, not stay at 11 — leaving the
> model falsely confident. To re-learn cleanly, re-run **§4.2** first to rebuild the causes
> from their priors, then §4.5, then this cell.


In [46]:
learned_posteriors = causal_api.learn(
    {
        "demand": [
            9.5,
            11.0,
            12.5,
            10.8,
            13.2,
            8.9,
            11.7,
            12.1,
            10.2,
            14.0,
        ],

        "market_growth": [
            0.01,
            0.03,
            -0.01,
            0.04,
            0.02,
            0.01,
            0.05,
            0.00,
            0.02,
            0.03,
        ],

        "competitor_active": [
            0,
            1,
            0,
            0,
            1,
            0,
            1,
            0,
            0,
            1,
        ],
    }
)

print(
    "Learned posterior distributions:"
)

print(
    learned_posteriors
)


Learned posterior distributions:
{'demand': {'distribution': 'Normal-Inverse-Gamma', 'posterior_mean': 11.263635635375977, 'kappa': 11.0, 'alpha': 8.0, 'beta': 20.682727813720703, 'expected_sampling_variance': 2.9546754019601003}, 'market_growth': {'distribution': 'Normal-Inverse-Gamma', 'posterior_mean': 0.019999997690320015, 'kappa': 12.0, 'alpha': 9.0, 'beta': 0.02149999886751175, 'expected_sampling_variance': 0.0026874998584389687}, 'competitor_active': {'distribution': 'Beta-Bernoulli', 'alpha': 6.0, 'beta': 9.0, 'posterior_probability_mean': 0.4, 'posterior_probability_variance': 0.015}}


### 4.7 Building the application

Compiles the LangGraph app against the engine and the `temperature=0` model.

`default_num_samples=20_000` is the width of every Monte Carlo estimate the app produces —
five thousand more than §2's default, and the main dial for trading runtime against
precision (standard error falls as `1/sqrt(n)`, so halving it costs 4× the samples).
`default_seed=123` makes every query in §5 reproducible and makes the two arms of an
intervention contrast paired.


In [47]:
app = build_causal_langgraph_app(
    causal_api=causal_api,
    model=model,
    default_num_samples=20_000,
    default_seed=123,
)


### 4.8 Inspecting the compiled graph

Prints the compiled state machine as Mermaid source — the ground truth for the diagram
drawn in §3. Worth an eye after any routing change: it shows the real edges, not the
intended ones. Paste the output into any Mermaid renderer to view it.


In [48]:
print(
    app.get_graph().draw_mermaid()
)


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	interpret_query(interpret_query)
	validate_query(validate_query)
	optimal_policy(optimal_policy)
	intervention_effect(intervention_effect)
	posterior_summary(posterior_summary)
	clarification(clarification)
	explain_result(explain_result)
	error(error)
	__end__([<p>__end__</p>]):::last
	__start__ --> interpret_query;
	interpret_query --> validate_query;
	intervention_effect -.-> error;
	intervention_effect -. &nbsp;explain&nbsp; .-> explain_result;
	optimal_policy -.-> error;
	optimal_policy -. &nbsp;explain&nbsp; .-> explain_result;
	posterior_summary -.-> error;
	posterior_summary -. &nbsp;explain&nbsp; .-> explain_result;
	validate_query -.-> clarification;
	validate_query -.-> error;
	validate_query -.-> intervention_effect;
	validate_query -.-> optimal_policy;
	validate_query -.-> posterior_summary;
	clarification --> __end__;
	error --> __end__;
	explain_result --> __end__;
	classDef 

### 4.9 `ask_causal_system` — the convenience wrapper

One call runs the whole pipeline and prints it in sections:

```text
    Gemini interpretation  ->  deterministic validation  ->  Pyro computation  ->  Gemini explanation
```

| Parameter | Effect |
| --- | --- |
| `question` | the natural-language input |
| `show_parsed_query=True` | prints the structured query — **the most useful debugging output**, since it shows exactly what the LLM understood |
| `show_numerical_result=False` | prints the raw authoritative result dict; on by default in §5 so the narration can be checked against the numbers |

It returns the full final state, so `result["numerical_result"]` and
`result["parsed_query"]` are available for further inspection after the printout.

When an answer looks wrong, read the parsed query first: nearly every surprising result
traces back to a value landing in `context` when the user meant an intervention, or a
variable being invented or dropped.

Note this wrapper closes over the module-level `app` — one global application, no session
concept. CHK-3/CHK-5 in `TICKETS.md` change that signature to take `app` and `session_id`
explicitly.


In [49]:
def ask_causal_system(
    question: str,
    show_parsed_query: bool = True,
    show_numerical_result: bool = False,
) -> dict:
    """
    Send a natural-language question through:

        Gemini query interpretation
            ->
        deterministic validation
            ->
        Pyro causal computation
            ->
        Gemini result explanation
    """

    result = app.invoke(
        {
            "user_input": question
        }
    )

    print("\n" + "=" * 80)
    print("USER QUESTION")
    print("=" * 80)
    print(question)

    if (
        show_parsed_query
        and result.get("parsed_query")
        is not None
    ):
        print("\n" + "=" * 80)
        print("LLM-PARSED QUERY")
        print("=" * 80)

        print(
            result[
                "parsed_query"
            ].model_dump_json(
                indent=2
            )
        )

    if (
        show_numerical_result
        and result.get(
            "numerical_result"
        )
        is not None
    ):
        print("\n" + "=" * 80)
        print("PYRO NUMERICAL RESULT")
        print("=" * 80)

        import json

        print(
            json.dumps(
                result[
                    "numerical_result"
                ],
                indent=2,
                default=str,
            )
        )

    print("\n" + "=" * 80)
    print("FINAL ANSWER")
    print("=" * 80)

    print(
        result.get(
            "final_answer",
            "No final answer was returned.",
        )
    )

    return result


---

# §5 Demo queries

Six questions, one per behaviour of the pipeline. Each makes **two live Gemini calls**
(interpret + explain), so this section needs a working key and network; §1–§4 do not.

| Demo | Question shape | Expected route | What it proves |
| --- | --- | --- | --- |
| 5.1 | partial information | `optimal_policy` | unstated causes stay stochastic |
| 5.2 | full information | `optimal_policy` | more context ⇒ narrower spread |
| 5.3 | forced change, plan held | `intervention_effect` (`fixed_policy`) | cost of a change under today's plan |
| 5.4 | forced change, plan adapts | `intervention_effect` (`reoptimise_policy`) | whether the best action itself moves |
| 5.5 | "what have you learned?" | `posterior_summary` | reads the posteriors, no sampling |
| 5.6 | deliberately ambiguous | `clarification_required` | refuses to guess |

Because §4.7 fixed the seed, re-running any cell reproduces the same numbers. The
*narration* may differ slightly between runs — Gemini's prose is not seeded — but the
numbers it is narrating cannot.

### 5.1 Optimal policy under partial information

Demand is stated; growth and competitor status are explicitly unknown. The correct
behaviour is for `context` to contain **only** `demand: 13.0`, leaving the other two to be
integrated over across all 20,000 worlds.

Check in the output: `context` has exactly one entry, and
`probability_each_action_is_best` is spread across actions rather than concentrated —
that spread *is* the residual uncertainty.


In [50]:
policy_result = ask_causal_system(
    """
    Demand is currently 13 units.

    I do not know whether the competitor is active and I do not
    know the market growth rate.

    Which policy should I choose to maximise expected utility?
    """,
    show_parsed_query=True,
    show_numerical_result=True,
)

print(policy_result)



USER QUESTION

    Demand is currently 13 units.

    I do not know whether the competitor is active and I do not
    know the market growth rate.

    Which policy should I choose to maximise expected utility?
    

FINAL ANSWER
The LLM could not convert the request into a structured causal query: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}
{'user_input': '\n    Demand is currently 13 units.\n\n    I do not know whether the competitor is active and I do not\n    know the market growth rate.\n\n    Which policy should I choose to maximise expected utility?\n    ', 'route': 'error', 'final_answer': "The LLM could not convert the request i

### 5.2 Optimal policy with full context

All three causes are stated, so every cause is clamped and only one world is effectively
simulated — all 20,000 draws are identical.

The tell: `utility_standard_deviation` collapses to ≈0 and `monte_carlo_standard_error`
with it, and one action takes essentially 100% of `probability_each_action_is_best`. That
is not the engine being suddenly confident about the future — it is the model being told
there is nothing left to be uncertain about.

Contrast with §5.1 to see what integrating over unknowns actually costs.


In [70]:
context_result = ask_causal_system(
    """
    Demand is 13 units, market growth is 0.04 and the competitor
    is active.

    Which decision maximises expected utility?
    """,
    show_parsed_query=True,
    show_numerical_result=True,
)

print(context_result)



USER QUESTION

    Demand is 13 units, market growth is 0.04 and the competitor
    is active.

    Which decision maximises expected utility?
    

FINAL ANSWER
The LLM could not convert the request into a structured causal query: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}
{'user_input': '\n    Demand is 13 units, market growth is 0.04 and the competitor\n    is active.\n\n    Which decision maximises expected utility?\n    ', 'route': 'error', 'final_answer': "The LLM could not convert the request into a structured causal query: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This

### 5.3 Fixed-policy intervention

"Force `competitor_active` to 1 rather than 0, **keeping decision 1 fixed**."

This is the estimand for a change you cannot respond to:

```text
    E[U | do(competitor_active=1), D=1]  -  E[U | do(competitor_active=0), D=1]
```

The phrasing supplies all three things `fixed_policy` needs: the intervention value (1),
an explicit baseline (0), and the decision to hold (1). Drop any one of them and §5.6's
clarification path is the correct response.

Expect a **negative** effect — an active competitor subtracts
`2 * max(effective_demand - 5, 0)`. With demand clamped at 13 and growth stochastic around
0.02, that is roughly −16 utility units.


In [71]:
fixed_intervention_result = (
    ask_causal_system(
        """
        Demand is currently 13 units.

        What is the causal effect on expected utility of forcing
        competitor_active to 1 rather than 0 while keeping
        decision 1 fixed?
        """,
        show_parsed_query=True,
        show_numerical_result=True,
    )
)



USER QUESTION

        Demand is currently 13 units.

        What is the causal effect on expected utility of forcing
        competitor_active to 1 rather than 0 while keeping
        decision 1 fixed?
        

FINAL ANSWER
The LLM could not convert the request into a structured causal query: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}


### 5.4 Reoptimised-policy intervention

The same contrast, but the decision is re-solved under each arm:

```text
    max_d E[U | do(competitor_active=1), D=d]  -  max_d E[U | do(competitor_active=0), D=d]
```

The result dict carries both `target_optimal_decision` and `baseline_optimal_decision`, so
you can see directly whether the best action moved.

**In this configuration it will not.** As §4.4 showed, `competitive_loss` is the same for
every decision, so it shifts all three utilities equally and cannot reorder them — this
effect should come out equal to §5.3's, with the optimal decision identical in both arms.
That is a correct result and a good sanity check on the engine, but it means this demo
does not yet exhibit the interesting case. To see the modes genuinely diverge, make the
competitor term interact with capacity, or reoptimise over a cause that already does —
`demand` is the natural candidate, since capacity choice is exactly what responds to it.


In [72]:
reoptimised_result = ask_causal_system(
    """
    Demand is currently 13 units.

    Compare forcing competitor_active to 1 against forcing it to 0.

    Reoptimise the decision under each intervention. What happens
    to the optimal decision and expected utility?
    """,
    show_parsed_query=True,
    show_numerical_result=True,
)



USER QUESTION

    Demand is currently 13 units.

    Compare forcing competitor_active to 1 against forcing it to 0.

    Reoptimise the decision under each intervention. What happens
    to the optimal decision and expected utility?
    

FINAL ANSWER
The LLM could not convert the request into a structured causal query: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}


### 5.5 Posterior summary

"What has the system learned?" — routes to `posterior_summary`, which reads the
hyperparameters straight off the cause objects. **No sampling happens**, so no seed and no
`num_samples` are involved and the result is exact.

The returned numbers should match the posterior table in §4.6. If `demand`'s `kappa` reads
21 rather than 11, §4.6 was run twice — see the warning there.


In [73]:
posterior_result = ask_causal_system(
    """
    What has the system learned about the marginal distributions
    of demand, market growth and competitor activity?
    """,
    show_parsed_query=True,
    show_numerical_result=True,
)



USER QUESTION

    What has the system learned about the marginal distributions
    of demand, market growth and competitor activity?
    

FINAL ANSWER
The LLM could not convert the request into a structured causal query: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}


### 5.6 Deliberately ambiguous query

"What happens if we force demand to change?" names an intervention verb and a real
variable, but never says **to what value**, nor against what baseline, nor whether the
decision should be held or reoptimised.

The correct behaviour is `clarification_required` with a specific follow-up question —
not a guessed value. This is the demo that tests the system's willingness to refuse: an
LLM that invents "let's say 15" would produce a fully-formed, precise, entirely
unfounded answer, which is the worst available failure mode for a decision tool.

The graph terminates here without a second LLM call and without touching Pyro. Note also
that the follow-up currently dead-ends — there is no conversation state to answer it into
(§3.4, CHK-3).


In [ ]:
clarification_result = ask_causal_system(
    """
    What happens if we force demand to change?
    """,
    show_parsed_query=True,
    show_numerical_result=True,
)


---

# Appendix — superseded implementation (do not run)

> **⚠ Dead code, kept for reference only. Running these cells breaks the live system.**

An earlier parallel implementation of the same idea, built around `CausalDecisionEngine`
and `build_application` instead of `CausalDecisionAPI` and `build_causal_langgraph_app`.
Nothing in §1–§5 references it, and `TICKETS.md` (CHK-1) explicitly excludes it from the
package extraction.

**Why running it is not harmless.** These cells rebind names the live system depends on:

| Name | Live definition | Appendix redefinition | Consequence |
| --- | --- | --- | --- |
| `Intervention` | §1.6 — has `mode` | dataclass with `reoptimise_policy: bool`, no `mode` | §3.6's `run_intervention_effect` raises `TypeError: unexpected keyword argument 'mode'` |
| `BayesianNormalCause` | §1.5 — accepts `description` | dataclass without a `description` field | re-running §4.2 raises `TypeError: unexpected keyword argument 'description'` |
| `BayesianBernoulliCause` | §1.3 | same problem | as above |
| `utility_function` | §4.4 | different implementation | a rebuilt engine silently optimises a different objective |
| `Decision`, `Tensor` | §1.1 | narrowed aliases | annotations only — harmless |

The failures are loud in the first three cases and **silent in the fourth**, which is the
dangerous one. If you have run any of this, restart the kernel and re-run from the top
rather than trying to patch it up.

Two candidate follow-ups, whichever suits: delete these cells outright once the package
extraction in CHK-1 lands, or move them to a `scratch/` notebook so the main one has one
implementation and no traps.


In [ ]:
Tensor = torch.Tensor
Decision = int | float | str
QueryType = Literal["optimal_policy", "intervention_effect"]


### Appendix A — `CauseModel` and `CausalDecisionEngine` (dead)

The earlier Layer 1 + Layer 2. Same conjugate mathematics as §1, expressed as
dataclasses, plus `CausalDecisionEngine` with `DecisionQuery`/`ActionEvaluation` value
objects where the live API passes plain dicts. Superseded by §1 and §2.


In [ ]:
class CauseModel:
    """
    Interface implemented by every independent cause model.

    Each cause is responsible for:
      1. learning its own marginal distribution;
      2. drawing posterior-predictive samples;
      3. reporting posterior information.
    """

    name: str

    def update(self, observations: list[float | int]) -> None:
        raise NotImplementedError

    def sample_posterior_predictive(
        self,
        num_samples: int,
    ) -> Tensor:
        raise NotImplementedError

    def posterior_summary(self) -> dict[str, float]:
        raise NotImplementedError


@dataclass
class BayesianNormalCause(CauseModel):
    """
    Bayesian Normal model with unknown mean and variance.

        X_i | mu, sigma^2 ~ Normal(mu, sigma^2)

    Conjugate prior:

        sigma^2 ~ InverseGamma(alpha_0, beta_0)
        mu | sigma^2 ~ Normal(mu_0, sigma^2 / kappa_0)

    We store the current posterior hyperparameters. Because the
    model is conjugate, updates are exact and do not require SVI.
    Pyro is used to sample the posterior and posterior predictive.
    """

    name: str
    mu: float = 0.0
    kappa: float = 1.0
    alpha: float = 2.0
    beta: float = 2.0

    def update(self, observations: list[float | int]) -> None:
        if not observations:
            return

        x = torch.as_tensor(observations, dtype=torch.float32)
        n = x.numel()

        sample_mean = x.mean()
        centred_sum_squares = ((x - sample_mean) ** 2).sum()

        old_mu = torch.tensor(self.mu)
        old_kappa = torch.tensor(self.kappa)
        old_alpha = torch.tensor(self.alpha)
        old_beta = torch.tensor(self.beta)

        new_kappa = old_kappa + n
        new_mu = (
            old_kappa * old_mu + n * sample_mean
        ) / new_kappa

        new_alpha = old_alpha + n / 2.0

        mean_adjustment = (
            old_kappa
            * n
            * (sample_mean - old_mu) ** 2
            / (2.0 * new_kappa)
        )

        new_beta = (
            old_beta
            + 0.5 * centred_sum_squares
            + mean_adjustment
        )

        self.mu = float(new_mu)
        self.kappa = float(new_kappa)
        self.alpha = float(new_alpha)
        self.beta = float(new_beta)

    def sample_posterior_predictive(
        self,
        num_samples: int,
    ) -> Tensor:
        alpha = torch.tensor(self.alpha)
        beta = torch.tensor(self.beta)
        mu = torch.tensor(self.mu)
        kappa = torch.tensor(self.kappa)

        # precision = 1 / variance
        precision = pyro.sample(
            f"{self.name}_precision",
            dist.Gamma(alpha, beta),
        )

        posterior_mu = pyro.sample(
            f"{self.name}_mu",
            dist.Normal(
                mu,
                torch.rsqrt(kappa * precision),
            ),
        )

        return pyro.sample(
            f"{self.name}_predictive",
            dist.Normal(
                posterior_mu,
                torch.rsqrt(precision),
            ).expand([num_samples]),
        )

    def posterior_summary(self) -> dict[str, float]:
        expected_variance = (
            self.beta / (self.alpha - 1.0)
            if self.alpha > 1.0
            else math.inf
        )

        return {
            "posterior_mean": self.mu,
            "expected_variance": expected_variance,
            "kappa": self.kappa,
            "alpha": self.alpha,
            "beta": self.beta,
        }


@dataclass
class BayesianBernoulliCause(CauseModel):
    """
    Bayesian Bernoulli model:

        X_i | theta ~ Bernoulli(theta)
        theta ~ Beta(alpha, beta)

    Posterior after s successes and f failures:

        theta | data ~ Beta(alpha + s, beta + f)
    """

    name: str
    alpha: float = 1.0
    beta: float = 1.0

    def update(self, observations: list[float | int]) -> None:
        if not observations:
            return

        values = torch.as_tensor(observations, dtype=torch.float32)

        if not torch.all((values == 0) | (values == 1)):
            raise ValueError(
                f"Bernoulli cause '{self.name}' requires 0/1 observations."
            )

        successes = values.sum().item()
        failures = values.numel() - successes

        self.alpha += successes
        self.beta += failures

    def sample_posterior_predictive(
        self,
        num_samples: int,
    ) -> Tensor:
        probability = pyro.sample(
            f"{self.name}_probability",
            dist.Beta(
                torch.tensor(self.alpha),
                torch.tensor(self.beta),
            ),
        )

        return pyro.sample(
            f"{self.name}_predictive",
            dist.Bernoulli(probability).expand([num_samples]),
        )

    def posterior_summary(self) -> dict[str, float]:
        total = self.alpha + self.beta
        mean = self.alpha / total
        variance = (
            self.alpha * self.beta
            / (total**2 * (total + 1.0))
        )

        return {
            "posterior_probability": mean,
            "posterior_probability_variance": variance,
            "alpha": self.alpha,
            "beta": self.beta,
        }


# ============================================================
# 2. QUERY AND RESPONSE OBJECTS
# ============================================================

@dataclass(frozen=True)
class Intervention:
    variable: str
    value: float | int

    # Optional baseline intervention for a causal contrast.
    baseline_value: float | int | None = None

    # If supplied, evaluate the intervention at this fixed action.
    fixed_decision: Decision | None = None

    # Otherwise choose the best decision under each intervention.
    reoptimise_policy: bool = False


@dataclass(frozen=True)
class DecisionQuery:
    query_type: QueryType

    # Observed subset X of C.
    context: dict[str, float | int] = field(default_factory=dict)

    intervention: Intervention | None = None

    num_samples: int = 10_000
    seed: int = 123


@dataclass
class ActionEvaluation:
    decision: Decision
    expected_utility: float
    utility_std: float
    monte_carlo_standard_error: float
    lower_95: float
    upper_95: float


# ============================================================
# 3. CAUSAL DECISION ENGINE
# ============================================================

class CausalDecisionEngine:
    """
    Represents:

        p(C_1, ..., C_p) = product_j p(C_j)
        U = utility_fn(D, C)

    Causes are independent marginal models.
    """

    def __init__(
        self,
        causes: dict[str, CauseModel],
        decisions: list[Decision],
        utility_fn: Callable[[Decision, dict[str, Tensor]], Tensor],
    ) -> None:
        if not causes:
            raise ValueError("At least one cause is required.")

        if not decisions:
            raise ValueError("At least one decision is required.")

        self.causes = causes
        self.decisions = decisions
        self.utility_fn = utility_fn

    # --------------------------------------------------------
    # Learning
    # --------------------------------------------------------

    def learn(
        self,
        data: dict[str, list[float | int]],
    ) -> dict[str, dict[str, float]]:
        """
        Update each marginal cause model independently.

        Under the independence assumption:

            p(theta_1, ..., theta_p | data)
            = product_j p(theta_j | data_j).
        """

        unknown = set(data) - set(self.causes)

        if unknown:
            raise ValueError(
                f"Unknown causes in learning data: {sorted(unknown)}"
            )

        for name, observations in data.items():
            self.causes[name].update(observations)

        return self.posterior_summaries()

    def posterior_summaries(self) -> dict[str, dict[str, float]]:
        return {
            name: model.posterior_summary()
            for name, model in self.causes.items()
        }

    # --------------------------------------------------------
    # Sampling the SCM
    # --------------------------------------------------------

    def sample_causes(
        self,
        num_samples: int,
        context: dict[str, float | int] | None = None,
        interventions: dict[str, float | int] | None = None,
    ) -> dict[str, Tensor]:
        """
        Generate samples from the model after applying:

          observations/context:
              C_j = observed value for this decision query;

          interventions:
              do(C_j = intervention value).

        In the current independent-root model, observations and
        interventions lead to the same numerical substitution for
        that variable. They remain conceptually different.
        """

        context = context or {}
        interventions = interventions or {}

        unknown_context = set(context) - set(self.causes)
        unknown_interventions = set(interventions) - set(self.causes)

        if unknown_context:
            raise ValueError(
                f"Unknown context variables: {sorted(unknown_context)}"
            )

        if unknown_interventions:
            raise ValueError(
                "Unknown intervention variables: "
                f"{sorted(unknown_interventions)}"
            )

        overlap = set(context) & set(interventions)

        for name in overlap:
            if context[name] != interventions[name]:
                raise ValueError(
                    f"Variable '{name}' is both observed as "
                    f"{context[name]} and intervened on as "
                    f"{interventions[name]}."
                )

        samples: dict[str, Tensor] = {}

        for name, cause_model in self.causes.items():
            if name in interventions:
                value = float(interventions[name])
                samples[name] = torch.full(
                    (num_samples,),
                    value,
                    dtype=torch.float32,
                )

            elif name in context:
                value = float(context[name])
                samples[name] = torch.full(
                    (num_samples,),
                    value,
                    dtype=torch.float32,
                )

            else:
                samples[name] = (
                    cause_model.sample_posterior_predictive(
                        num_samples
                    )
                )

        return samples

    # --------------------------------------------------------
    # Action evaluation
    # --------------------------------------------------------

    def evaluate_actions(
        self,
        context: dict[str, float | int] | None = None,
        interventions: dict[str, float | int] | None = None,
        num_samples: int = 10_000,
        seed: int = 123,
    ) -> tuple[list[ActionEvaluation], dict[Decision, Tensor]]:
        """
        Evaluate every decision using common random numbers.

        The same sampled causal worlds are used for every action.
        This reduces Monte Carlo noise when comparing actions.
        """

        pyro.set_rng_seed(seed)

        causes = self.sample_causes(
            num_samples=num_samples,
            context=context,
            interventions=interventions,
        )

        evaluations: list[ActionEvaluation] = []
        utilities: dict[Decision, Tensor] = {}

        for decision in self.decisions:
            utility_samples = self.utility_fn(
                decision,
                causes,
            ).reshape(-1)

            if utility_samples.numel() != num_samples:
                raise ValueError(
                    "utility_fn must return one utility per sample. "
                    f"Expected {num_samples}, received "
                    f"{utility_samples.numel()}."
                )

            utility_samples = utility_samples.detach()
            utilities[decision] = utility_samples

            mean = utility_samples.mean()
            std = utility_samples.std(unbiased=True)
            standard_error = std / math.sqrt(num_samples)

            evaluations.append(
                ActionEvaluation(
                    decision=decision,
                    expected_utility=float(mean),
                    utility_std=float(std),
                    monte_carlo_standard_error=float(
                        standard_error
                    ),
                    lower_95=float(
                        mean - 1.96 * standard_error
                    ),
                    upper_95=float(
                        mean + 1.96 * standard_error
                    ),
                )
            )

        return evaluations, utilities

    def optimal_policy(
        self,
        context: dict[str, float | int] | None = None,
        interventions: dict[str, float | int] | None = None,
        num_samples: int = 10_000,
        seed: int = 123,
    ) -> dict[str, Any]:
        evaluations, utilities = self.evaluate_actions(
            context=context,
            interventions=interventions,
            num_samples=num_samples,
            seed=seed,
        )

        best = max(
            evaluations,
            key=lambda item: item.expected_utility,
        )

        # For each sampled causal world, determine which action
        # produced the highest realised utility.
        utility_matrix = torch.stack(
            [utilities[d] for d in self.decisions],
            dim=1,
        )

        winning_indices = utility_matrix.argmax(dim=1)

        probability_best = {
            decision: float(
                (winning_indices == index).float().mean()
            )
            for index, decision in enumerate(self.decisions)
        }

        return {
            "optimal_decision": best.decision,
            "optimal_expected_utility": best.expected_utility,
            "action_evaluations": [
                evaluation.__dict__
                for evaluation in evaluations
            ],
            "probability_each_action_is_best": probability_best,
            "context": context or {},
            "interventions": interventions or {},
        }

    # --------------------------------------------------------
    # Intervention analysis
    # --------------------------------------------------------

    def intervention_effect(
        self,
        intervention: Intervention,
        context: dict[str, float | int] | None = None,
        num_samples: int = 10_000,
        seed: int = 123,
    ) -> dict[str, Any]:
        if intervention.variable not in self.causes:
            raise ValueError(
                f"Cannot intervene on unknown variable "
                f"'{intervention.variable}'."
            )

        if (
            intervention.fixed_decision is not None
            and intervention.reoptimise_policy
        ):
            raise ValueError(
                "Choose either fixed_decision or "
                "reoptimise_policy, not both."
            )

        if (
            intervention.fixed_decision is None
            and not intervention.reoptimise_policy
        ):
            raise ValueError(
                "An intervention query must specify either "
                "fixed_decision or reoptimise_policy=True."
            )

        if (
            intervention.fixed_decision is not None
            and intervention.fixed_decision not in self.decisions
        ):
            raise ValueError(
                f"Unknown decision: {intervention.fixed_decision}"
            )

        target_do = {
            intervention.variable: intervention.value
        }

        baseline_do = (
            {
                intervention.variable:
                intervention.baseline_value
            }
            if intervention.baseline_value is not None
            else {}
        )

        if intervention.reoptimise_policy:
            target = self.optimal_policy(
                context=context,
                interventions=target_do,
                num_samples=num_samples,
                seed=seed,
            )

            baseline = self.optimal_policy(
                context=context,
                interventions=baseline_do,
                num_samples=num_samples,
                seed=seed,
            )

            contrast = (
                target["optimal_expected_utility"]
                - baseline["optimal_expected_utility"]
            )

            return {
                "estimand": "reoptimised_policy_effect",
                "variable": intervention.variable,
                "intervention_value": intervention.value,
                "baseline_value": intervention.baseline_value,
                "target_optimal_decision":
                    target["optimal_decision"],
                "baseline_optimal_decision":
                    baseline["optimal_decision"],
                "target_expected_utility":
                    target["optimal_expected_utility"],
                "baseline_expected_utility":
                    baseline["optimal_expected_utility"],
                "causal_effect": contrast,
                "target": target,
                "baseline": baseline,
            }

        decision = intervention.fixed_decision

        target_evaluations, _ = self.evaluate_actions(
            context=context,
            interventions=target_do,
            num_samples=num_samples,
            seed=seed,
        )

        baseline_evaluations, _ = self.evaluate_actions(
            context=context,
            interventions=baseline_do,
            num_samples=num_samples,
            seed=seed,
        )

        target = next(
            item
            for item in target_evaluations
            if item.decision == decision
        )

        baseline = next(
            item
            for item in baseline_evaluations
            if item.decision == decision
        )

        return {
            "estimand": "fixed_policy_effect",
            "decision": decision,
            "variable": intervention.variable,
            "intervention_value": intervention.value,
            "baseline_value": intervention.baseline_value,
            "target_expected_utility":
                target.expected_utility,
            "baseline_expected_utility":
                baseline.expected_utility,
            "causal_effect": (
                target.expected_utility
                - baseline.expected_utility
            ),
            "target_evaluation": target.__dict__,
            "baseline_evaluation": baseline.__dict__,
        }


### Appendix B — `build_application` and the old example (dead)

The earlier Layer 3: a LangGraph app with **no LLM at all** — it took a `DecisionQuery`
object directly, so there was no interpretation node and no explanation node. The live
pipeline in §3 added both ends. `create_example_system()` is the ancestor of §4.
Superseded by §3 and §4.


In [ ]:
class ApplicationState(TypedDict, total=False):
    query: DecisionQuery
    validated_query: DecisionQuery
    route: QueryType
    result: dict[str, Any]
    error: str


# ============================================================
# 5. LANGGRAPH APPLICATION
# ============================================================

def build_application(
    engine: CausalDecisionEngine,
):
    def validate_query(
        state: ApplicationState,
    ) -> ApplicationState:
        query = state["query"]

        if query.num_samples <= 1:
            return {
                "error": "num_samples must be greater than one."
            }

        unknown_context = (
            set(query.context) - set(engine.causes)
        )

        if unknown_context:
            return {
                "error": (
                    "Unknown context variables: "
                    f"{sorted(unknown_context)}"
                )
            }

        if (
            query.query_type == "intervention_effect"
            and query.intervention is None
        ):
            return {
                "error": (
                    "intervention_effect requires an "
                    "Intervention object."
                )
            }

        return {
            "validated_query": query,
            "route": query.query_type,
        }

    def route_after_validation(
        state: ApplicationState,
    ) -> str:
        if "error" in state:
            return "error"

        return state["route"]

    def optimal_policy_node(
        state: ApplicationState,
    ) -> ApplicationState:
        query = state["validated_query"]

        result = engine.optimal_policy(
            context=query.context,
            num_samples=query.num_samples,
            seed=query.seed,
        )

        return {"result": result}

    def intervention_node(
        state: ApplicationState,
    ) -> ApplicationState:
        query = state["validated_query"]

        assert query.intervention is not None

        result = engine.intervention_effect(
            intervention=query.intervention,
            context=query.context,
            num_samples=query.num_samples,
            seed=query.seed,
        )

        return {"result": result}

    def error_node(
        state: ApplicationState,
    ) -> ApplicationState:
        return {
            "result": {
                "status": "error",
                "message": state["error"],
            }
        }

    builder = StateGraph(ApplicationState)

    builder.add_node("validate_query", validate_query)
    builder.add_node(
        "optimal_policy",
        optimal_policy_node,
    )
    builder.add_node(
        "intervention_effect",
        intervention_node,
    )
    builder.add_node("error", error_node)

    builder.add_edge(START, "validate_query")

    builder.add_conditional_edges(
        "validate_query",
        route_after_validation,
        {
            "optimal_policy": "optimal_policy",
            "intervention_effect":
                "intervention_effect",
            "error": "error",
        },
    )

    builder.add_edge("optimal_policy", END)
    builder.add_edge("intervention_effect", END)
    builder.add_edge("error", END)

    return builder.compile()


# ============================================================
# 6. EXAMPLE UTILITY FUNCTION
# ============================================================

def utility_function(
    decision: Decision,
    causes: dict[str, Tensor],
) -> Tensor:
    """
    Example decision problem.

    D:
        0 = conservative policy
        1 = balanced policy
        2 = aggressive policy

    Causes:
        demand              continuous
        market_growth       continuous
        competitor_active   binary

    Deterministic utility:

        U = revenue - cost - competitive_loss - risk_penalty
    """

    demand = causes["demand"]
    growth = causes["market_growth"]
    competitor = causes["competitor_active"]

    if decision == 0:
        capacity = 8.0
        unit_margin = 6.0
        fixed_cost = 8.0
        risk_coefficient = 0.5

    elif decision == 1:
        capacity = 14.0
        unit_margin = 7.0
        fixed_cost = 18.0
        risk_coefficient = 1.2

    elif decision == 2:
        capacity = 22.0
        unit_margin = 8.0
        fixed_cost = 35.0
        risk_coefficient = 2.5

    else:
        raise ValueError(f"Unknown decision: {decision}")

    effective_demand = demand * (1.0 + growth)

    units_sold = torch.minimum(
        effective_demand,
        torch.tensor(capacity),
    )

    revenue = unit_margin * units_sold

    competitive_loss = (
        competitor
        * 2.0
        * torch.clamp(effective_demand - 5.0, min=0.0)
    )

    excess_capacity = torch.clamp(
        torch.tensor(capacity) - effective_demand,
        min=0.0,
    )

    risk_penalty = risk_coefficient * excess_capacity

    return (
        revenue
        - fixed_cost
        - competitive_loss
        - risk_penalty
    )


# ============================================================
# 7. CONSTRUCT THE SYSTEM
# ============================================================

def create_example_system():
    causes: dict[str, CauseModel] = {
        "demand": BayesianNormalCause(
            name="demand",
            mu=10.0,
            kappa=1.0,
            alpha=3.0,
            beta=8.0,
        ),
        "market_growth": BayesianNormalCause(
            name="market_growth",
            mu=0.02,
            kappa=2.0,
            alpha=4.0,
            beta=0.02,
        ),
        "competitor_active": BayesianBernoulliCause(
            name="competitor_active",
            alpha=2.0,
            beta=3.0,
        ),
    }

    engine = CausalDecisionEngine(
        causes=causes,
        decisions=[0, 1, 2],
        utility_fn=utility_function,
    )

    application = build_application(engine)

    return engine, application


# ============================================================
# 8. EXAMPLE USE
# ============================================================

if __name__ == "__main__":
    engine, app = create_example_system()

    # --------------------------------------------------------
    # Learn marginal cause distributions from historical data.
    # --------------------------------------------------------

    posterior = engine.learn(
        {
            "demand": [
                9.5, 11.0, 12.5, 10.8, 13.2,
                8.9, 11.7, 12.1, 10.2, 14.0,
            ],
            "market_growth": [
                0.01, 0.03, -0.01, 0.04, 0.02,
                0.01, 0.05, 0.00, 0.02, 0.03,
            ],
            "competitor_active": [
                0, 1, 0, 0, 1,
                0, 1, 0, 0, 1,
            ],
        }
    )

    print("\nPosterior distributions")
    print(posterior)

    # --------------------------------------------------------
    # Query 1:
    # Find the optimal policy given dynamic context.
    #
    # Here demand is observed, while market growth and
    # competitor activity remain stochastic.
    # --------------------------------------------------------

    policy_query = DecisionQuery(
        query_type="optimal_policy",
        context={
            "demand": 13.0,
        },
        num_samples=20_000,
        seed=101,
    )

    policy_result = app.invoke(
        {"query": policy_query}
    )

    print("\nOptimal policy query")
    print(policy_result["result"])

    # --------------------------------------------------------
    # Query 2:
    # Fixed-policy causal effect.
    #
    # Compare:
    #
    #   E[U | do(competitor_active = 1), D = 1]
    #
    # against:
    #
    #   E[U | do(competitor_active = 0), D = 1].
    # --------------------------------------------------------

    fixed_intervention_query = DecisionQuery(
        query_type="intervention_effect",
        context={
            "demand": 13.0,
        },
        intervention=Intervention(
            variable="competitor_active",
            value=1,
            baseline_value=0,
            fixed_decision=1,
            reoptimise_policy=False,
        ),
        num_samples=20_000,
        seed=202,
    )

    fixed_result = app.invoke(
        {"query": fixed_intervention_query}
    )

    print("\nFixed-policy intervention")
    print(fixed_result["result"])

    # --------------------------------------------------------
    # Query 3:
    # Reoptimised intervention effect.
    #
    # Compare the maximum expected utility under:
    #
    #   do(competitor_active = 1)
    #
    # against:
    #
    #   do(competitor_active = 0).
    # --------------------------------------------------------

    adaptive_intervention_query = DecisionQuery(
        query_type="intervention_effect",
        context={
            "demand": 13.0,
        },
        intervention=Intervention(
            variable="competitor_active",
            value=1,
            baseline_value=0,
            reoptimise_policy=True,
        ),
        num_samples=20_000,
        seed=303,
    )

    adaptive_result = app.invoke(
        {"query": adaptive_intervention_query}
    )

    print("\nReoptimised intervention")
    print(adaptive_result["result"])
